# arXiv Research Articles Dataset — Binoculars Pipeline

**Run environment:** Google Colab with A100 GPU. Mount Drive before running.

**Required Colab Secrets** (Tools → Secrets):
- `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` — for `s3://arxiv` (requester-pays).
- `ANTHROPIC_API_KEY` and/or `OPENAI_API_KEY` — only needed for the positive-control rewrite cells (~cell 33+).

**What this notebook does:** downloads arXiv source tars from S3 → strips LaTeX → 250-word chunks → scores with Binoculars (Falcon-7B / Falcon-7B-Instruct, threshold `0.9015310749276843`) → produces the December 2021 vs December 2025 comparison from the report.

**Drive layout it expects:** `/content/drive/MyDrive/arxiv/` mirroring `gdrive_data/` from the repo's Drive folder.


Mounting Google Drive to Use Its Storage and Downloading rclone to copy from aws s3 bucket

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install rclone
!curl https://rclone.org/install.sh | sudo bash

#Dataset Download Code

In [ ]:
!pip install awscli -q

import os
from google.colab import userdata  # in Colab; locally use python-dotenv or os.environ

# REDACTED — supply your own AWS credentials via Colab Secrets (or env vars locally).
# Original keys were rotated and removed before publishing this repo.
os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'


In [ ]:
%%bash
# REDACTED — original AWS keys removed. The cell below builds an rclone config
# from environment variables set in the previous cell. Do NOT paste keys inline.
mkdir -p ~/.config/rclone
cat > ~/.config/rclone/rclone.conf << EOF
[s3]
type = s3
provider = AWS
access_key_id = ${AWS_ACCESS_KEY_ID}
secret_access_key = ${AWS_SECRET_ACCESS_KEY}
region = us-east-1
requester_pays = true
EOF


In [ ]:
%%bash
rclone listremotes

In [ ]:
%%bash
cat ~/.config/rclone/rclone.conf

In [ ]:
%%bash
rclone copy s3:arxiv/src/ /content/drive/MyDrive/arxivData/may24/ \
  --include "arXiv_src_2405_*.tar" \
  --progress \
  --s3-requester-pays

In [ ]:
!rclone copy s3:arxiv/src/ gdrive:arxiv/src/ --include "arXiv_src_2512_*.tar" --progress --s3-requester-pays

# Data Parsing Code
## 1. Configure Working Dir etc.

In [ ]:
"""
arXiv LaTeX → Clean 256-Word Chunks Preprocessing Pipeline
===========================================================
Processes ALL .tar files in a specified directory.
Designed to run in Google Colab with Google Drive mounted.

Usage:
    1. Mount Google Drive in Colab
    2. Set TAR_DIR to your directory containing .tar files
    3. Run all cells

Dependencies (install in Colab):
    !pip install chardet
"""

# =============================================================================
# CELL 1: Configuration & Imports
# =============================================================================

import os
import re
import json
import tarfile
import gzip
import shutil
import hashlib
import chardet
import warnings
import random
import csv
import time
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Optional, Tuple

# --- CONFIGURATION ---
# Directory containing .tar files on Google Drive
TAR_DIR = "/content/drive/MyDrive/arxivData/may24"  # <-- CHANGE THIS [DONE]

# Output directory for processed chunks
OUTPUT_DIR = "/content/drive/MyDrive/arxiv/processed/may_2024"  # <-- CHANGE THIS

# Working directory (local to Colab instance, faster I/O than Drive)
WORK_DIR = "/content/arxiv_work"

# Chunking parameters
CHUNK_WORD_COUNT = 256
CHUNKS_PER_PAPER = 4
MIN_BODY_WORDS = 1000  # Skip papers with fewer words than this
RANDOM_SEED = 42

# Create directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

random.seed(RANDOM_SEED)

# Find all .tar files in the directory
tar_files = sorted([
    os.path.join(TAR_DIR, f)
    for f in os.listdir(TAR_DIR)
    if f.endswith(".tar")
])

print(f"Found {len(tar_files)} tar files in {TAR_DIR}:")
for tf in tar_files:
    size_mb = os.path.getsize(tf) / 1024 / 1024
    print(f"  {os.path.basename(tf)} ({size_mb:.0f} MB)")
print(f"\nOutput dir: {OUTPUT_DIR}")
print(f"Work dir: {WORK_DIR}")
print(f"Chunk size: {CHUNK_WORD_COUNT} words, {CHUNKS_PER_PAPER} chunks per paper")


## 2. Defining all the processing functions

In [ ]:
# =============================================================================
# CELL 2: Define all processing functions
# =============================================================================

def extract_tar(tar_path: str, dest_dir: str) -> List[str]:
    """Extract the arXiv tar file. Returns list of extracted paper directories/files."""
    extract_dir = os.path.join(dest_dir, "raw_extracted")
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)
    os.makedirs(extract_dir)

    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(extract_dir)

    # The tar often extracts to a single subdirectory that contains
    # the actual .gz paper files. Walk the tree to find all paper files.
    papers = []
    for root, dirs, files in os.walk(extract_dir):
        for f in sorted(files):
            filepath = os.path.join(root, f)
            if f.endswith(".gz") or f.endswith(".pdf") or f.endswith(".tex"):
                papers.append(filepath)

    # If no individual files found, check if there are subdirectories
    # that represent individual papers
    if not papers:
        for root, dirs, files in os.walk(extract_dir):
            for d in sorted(dirs):
                dirpath = os.path.join(root, d)
                has_tex = any(
                    ff.endswith(".tex")
                    for ff in os.listdir(dirpath)
                    if os.path.isfile(os.path.join(dirpath, ff))
                )
                if has_tex:
                    papers.append(dirpath)

    return papers


def unpack_paper(paper_path: str, dest_dir: str) -> Optional[str]:
    """Unpack a single paper submission into a directory."""
    paper_id = os.path.basename(paper_path).replace(".gz", "")
    paper_dir = os.path.join(dest_dir, paper_id)
    os.makedirs(paper_dir, exist_ok=True)

    try:
        if paper_path.endswith(".gz"):
            with gzip.open(paper_path, "rb") as gz:
                content = gz.read()

            temp_path = os.path.join(paper_dir, "_temp_decompressed")
            with open(temp_path, "wb") as f:
                f.write(content)

            if tarfile.is_tarfile(temp_path):
                with tarfile.open(temp_path, "r") as tar:
                    tar.extractall(paper_dir)
                os.remove(temp_path)
            else:
                os.rename(temp_path, os.path.join(paper_dir, paper_id + ".tex"))

        elif os.path.isdir(paper_path):
            shutil.copytree(paper_path, paper_dir, dirs_exist_ok=True)

        elif os.path.isfile(paper_path):
            shutil.copy2(paper_path, paper_dir)

        return paper_dir

    except Exception as e:
        return None


def read_file_with_encoding(filepath: str) -> Optional[str]:
    """Read a file, handling various encodings."""
    try:
        with open(filepath, "rb") as f:
            raw = f.read()

        try:
            return raw.decode("utf-8")
        except UnicodeDecodeError:
            pass

        detected = chardet.detect(raw)
        if detected["encoding"]:
            try:
                return raw.decode(detected["encoding"])
            except (UnicodeDecodeError, LookupError):
                pass

        return raw.decode("latin-1")

    except Exception:
        return None


def find_main_tex(paper_dir: str) -> Optional[str]:
    """Find the main .tex file in a paper directory."""
    tex_files = []
    for root, dirs, files in os.walk(paper_dir):
        for f in files:
            if f.endswith(".tex"):
                tex_files.append(os.path.join(root, f))

    if not tex_files:
        return None

    if len(tex_files) == 1:
        return tex_files[0]

    candidates = []
    for tf in tex_files:
        content = read_file_with_encoding(tf)
        if content and re.search(r"\\documentclass", content):
            candidates.append(tf)

    if not candidates:
        candidates = tex_files

    if len(candidates) == 1:
        return candidates[0]

    preferred_names = ["main.tex", "paper.tex", "article.tex", "manuscript.tex"]
    for name in preferred_names:
        for c in candidates:
            if os.path.basename(c).lower() == name:
                return c

    return max(candidates, key=os.path.getsize)


def resolve_inputs(content: str, base_dir: str, depth: int = 0) -> str:
    """Resolve \\input{} and \\include{} commands by inlining referenced files."""
    if depth > 10:
        return content

    def replace_input(match):
        filename = match.group(1).strip()
        if not filename.endswith(".tex"):
            filename += ".tex"

        filepath = os.path.join(base_dir, filename)
        if os.path.exists(filepath):
            included = read_file_with_encoding(filepath)
            if included:
                return resolve_inputs(included, os.path.dirname(filepath), depth + 1)
        return ""

    pattern = r"\\(?:input|include)\s*\{([^}]+)\}"
    return re.sub(pattern, replace_input, content)


def strip_latex_to_text(latex: str) -> str:
    """Convert LaTeX to plain text by stripping commands and environments."""
    text = latex

    # Remove comments
    text = re.sub(r"(?<!\\)%.*$", "", text, flags=re.MULTILINE)

    # Keep only content between \begin{document} and \end{document}
    doc_match = re.search(r"\\begin\{document\}", text)
    if doc_match:
        text = text[doc_match.end():]
    end_match = re.search(r"\\end\{document\}", text)
    if end_match:
        text = text[:end_match.start()]

    # Remove bibliography/references
    text = re.sub(
        r"\\(?:section|chapter)\*?\{(?:References|Bibliography|Works Cited)\}.*",
        "", text, flags=re.DOTALL | re.IGNORECASE
    )
    text = re.sub(r"\\bibliography\{[^}]*\}", "", text)
    text = re.sub(r"\\bibliographystyle\{[^}]*\}", "", text)
    text = re.sub(
        r"\\begin\{thebibliography\}.*?\\end\{thebibliography\}",
        "", text, flags=re.DOTALL
    )

    # Remove acknowledgments
    text = re.sub(
        r"\\(?:section|paragraph)\*?\{Acknowledg[e]?ments?\}.*?(?=\\(?:section|chapter|appendix|end\{document\})|$)",
        "", text, flags=re.DOTALL | re.IGNORECASE
    )

    # Remove appendix content
    text = re.sub(r"\\appendix.*", "", text, flags=re.DOTALL)

    # Remove abstract
    text = re.sub(r"\\begin\{abstract\}.*?\\end\{abstract\}", "", text, flags=re.DOTALL)

    # Remove non-prose environments
    envs_to_remove = [
        "figure", "figure\\*", "table", "table\\*", "tabular", "tabular\\*",
        "equation", "equation\\*", "align", "align\\*", "eqnarray", "eqnarray\\*",
        "gather", "gather\\*", "multline", "multline\\*",
        "lstlisting", "verbatim", "minted", "algorithm", "algorithmic",
        "tikzpicture", "pgfpicture",
    ]
    for env in envs_to_remove:
        env_escaped = re.escape(env)
        text = re.sub(
            rf"\\begin\{{{env_escaped}\}}.*?\\end\{{{env_escaped}\}}",
            " ", text, flags=re.DOTALL
        )

    # Remove math
    text = re.sub(r"\\\[.*?\\\]", " ", text, flags=re.DOTALL)
    text = re.sub(r"\$\$.*?\$\$", " ", text, flags=re.DOTALL)
    text = re.sub(r"(?<!\$)\$(?!\$)(.+?)(?<!\$)\$(?!\$)", " ", text)

    # Remove citations, refs, labels
    text = re.sub(r"\\(?:cite|citep|citet|ref|eqref|label|pageref|autoref|Cref|cref)\w*\{[^}]*\}", "", text)

    # Extract text from formatting commands
    text = re.sub(r"\\footnote\{([^}]*)\}", r" \1 ", text)
    text = re.sub(r"\\(?:textbf|textit|emph|texttt|textrm|textsf|textsc|underline|mbox)\{([^}]*)\}", r"\1", text)
    text = re.sub(r"\\(?:section|subsection|subsubsection|paragraph|subparagraph)\*?\{([^}]*)\}", r"\n\1\n", text)

    # Remove remaining commands
    text = re.sub(r"\\[a-zA-Z]+\*?(?:\[[^\]]*\])*(?:\{[^}]*\})*", " ", text)

    # Remove braces and leftover symbols
    text = re.sub(r"[{}]", "", text)
    text = re.sub(r"\\[&%$#_~^]", " ", text)
    text = re.sub(r"~", " ", text)

    # Clean whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = text.strip()

    return text


def is_english(text: str, threshold: float = 0.7) -> bool:
    """Simple heuristic to check if text is primarily English."""
    words = text.split()
    if len(words) < 50:
        return True

    ascii_words = sum(1 for w in words if all(ord(c) < 128 for c in w))
    ratio = ascii_words / len(words)

    common_english = {"the", "a", "an", "is", "are", "was", "were", "in", "of", "to", "and", "for", "that", "this", "with"}
    lower_words = set(w.lower() for w in words[:200])
    english_word_count = len(common_english & lower_words)

    return ratio > threshold and english_word_count >= 5


def find_sentence_boundary(text: str, target_pos: int, direction: str = "forward") -> int:
    """Find the nearest sentence boundary to target_pos."""
    sentence_enders = re.compile(r"[.!?]\s")

    if direction == "forward":
        match = sentence_enders.search(text, target_pos)
        if match:
            return match.end()
        return len(text)
    else:
        substring = text[:target_pos]
        matches = list(sentence_enders.finditer(substring))
        if matches:
            return matches[-1].end()
        return 0


def create_chunks(
    text: str,
    chunk_word_count: int = 256,
    num_chunks: int = 4,
    tolerance: float = 0.15,
) -> List[Dict]:
    """Sample non-overlapping chunks from the text body."""
    words = text.split()
    total_words = len(words)

    if total_words < chunk_word_count * num_chunks:
        num_chunks = max(1, total_words // chunk_word_count)

    if num_chunks == 0:
        return []

    zone_size = total_words // num_chunks
    chunks = []

    min_words = int(chunk_word_count * (1 - tolerance))
    max_words = int(chunk_word_count * (1 + tolerance))

    for i in range(num_chunks):
        zone_start_word = i * zone_size
        zone_end_word = min((i + 1) * zone_size, total_words)

        latest_start = zone_end_word - chunk_word_count
        if latest_start <= zone_start_word:
            latest_start = zone_start_word

        start_word = random.randint(zone_start_word, latest_start)

        start_char = sum(len(w) + 1 for w in words[:start_word])
        end_word_idx = min(start_word + chunk_word_count, total_words)
        end_char = sum(len(w) + 1 for w in words[:end_word_idx])

        snapped_start = find_sentence_boundary(text, max(0, start_char - 1), "backward")
        snapped_end = find_sentence_boundary(text, end_char, "forward")

        chunk_text = text[snapped_start:snapped_end].strip()
        chunk_words = len(chunk_text.split())

        if chunk_words > max_words:
            chunk_words_list = chunk_text.split()
            truncated = " ".join(chunk_words_list[:chunk_word_count])
            boundary = find_sentence_boundary(truncated, len(truncated) - 1, "backward")
            if boundary > len(truncated) * 0.5:
                chunk_text = truncated[:boundary].strip()

        chunk_words = len(chunk_text.split())

        if chunk_words < min_words:
            continue

        chunks.append({
            "text": chunk_text,
            "word_count": chunk_words,
            "zone": i,
            "start_word_approx": start_word,
        })

    return chunks


def process_single_tar(tar_path: str, work_dir: str) -> Tuple[List[Dict], Dict]:
    """
    Process a single tar file end-to-end.
    Returns (list of chunks, stats dict).
    """
    tar_name = os.path.basename(tar_path)
    stats = {
        "tar_file": tar_name,
        "total_items": 0,
        "unpacked": 0,
        "tex_found": 0,
        "processed": 0,
        "papers_with_chunks": 0,
        "total_chunks": 0,
        "skipped_reasons": Counter(),
    }

    # --- Extract tar ---
    raw_papers = extract_tar(tar_path, work_dir)
    stats["total_items"] = len(raw_papers)
    extensions = Counter(os.path.splitext(p)[1] if os.path.isfile(p) else "dir" for p in raw_papers)
    print(f"    Items: {len(raw_papers)} | Types: {dict(extensions)}")

    # --- Unpack each paper ---
    unpacked_dir = os.path.join(work_dir, "unpacked")
    if os.path.exists(unpacked_dir):
        shutil.rmtree(unpacked_dir)
    os.makedirs(unpacked_dir)

    paper_dirs = []
    for p in raw_papers:
        result = unpack_paper(p, unpacked_dir)
        if result:
            paper_dirs.append(result)
    stats["unpacked"] = len(paper_dirs)
    print(f"    Unpacked: {len(paper_dirs)}")

    # --- Find main .tex files ---
    paper_tex_map = {}
    for pdir in paper_dirs:
        paper_id = os.path.basename(pdir)
        main_tex = find_main_tex(pdir)
        if main_tex:
            paper_tex_map[paper_id] = main_tex
    stats["tex_found"] = len(paper_tex_map)
    print(f"    Found .tex: {len(paper_tex_map)}")

    # --- Convert LaTeX to text and create chunks ---
    all_chunks = []

    for paper_id, tex_path in paper_tex_map.items():
        try:
            content = read_file_with_encoding(tex_path)
            if not content:
                stats["skipped_reasons"]["unreadable_file"] += 1
                continue

            base_dir = os.path.dirname(tex_path)
            content = resolve_inputs(content, base_dir)
            plain_text = strip_latex_to_text(content)

            words = plain_text.split()
            if len(words) < MIN_BODY_WORDS:
                stats["skipped_reasons"]["too_short"] += 1
                continue

            if not is_english(plain_text):
                stats["skipped_reasons"]["non_english"] += 1
                continue

            stats["processed"] += 1

            chunks = create_chunks(
                plain_text,
                chunk_word_count=CHUNK_WORD_COUNT,
                num_chunks=CHUNKS_PER_PAPER,
            )

            if chunks:
                stats["papers_with_chunks"] += 1
                for idx, chunk in enumerate(chunks):
                    all_chunks.append({
                        "paper_id": paper_id,
                        "source_tar": tar_name,
                        "chunk_index": idx,
                        "text": chunk["text"],
                        "word_count": chunk["word_count"],
                        "zone": chunk["zone"],
                    })

        except Exception as e:
            stats["skipped_reasons"]["processing_error"] += 1
            continue

    stats["total_chunks"] = len(all_chunks)
    print(f"    Processed: {stats['processed']} | Chunks: {stats['total_chunks']}")

    if stats["skipped_reasons"]:
        for reason, count in stats["skipped_reasons"].most_common():
            print(f"      Skipped — {reason}: {count}")

    return all_chunks, stats

##3. Processing all the .tar files

In [ ]:
# =============================================================================
# CELL 3: Process all tar files
# =============================================================================

all_chunks_combined = []
all_stats = []
total_start_time = time.time()

# Check for existing progress (resume support)
progress_path = os.path.join(OUTPUT_DIR, "_progress.json")
completed_tars = set()
if os.path.exists(progress_path):
    with open(progress_path, "r") as f:
        progress = json.load(f)
    completed_tars = set(progress.get("completed", []))
    print(f"Resuming — {len(completed_tars)} tar files already processed")
    # Load previously saved chunks
    combined_path = os.path.join(OUTPUT_DIR, "all_chunks_combined.json")
    if os.path.exists(combined_path):
        with open(combined_path, "r") as f:
            existing = json.load(f)
        all_chunks_combined = existing.get("chunks", [])
        all_stats = existing.get("per_tar_stats", [])
        print(f"Loaded {len(all_chunks_combined)} existing chunks")

for tar_idx, tar_path in enumerate(tar_files):
    tar_name = os.path.basename(tar_path)

    # Skip if already processed
    if tar_name in completed_tars:
        print(f"\n[{tar_idx + 1}/{len(tar_files)}] SKIPPING {tar_name} (already done)")
        continue

    print(f"\n{'='*70}")
    print(f"[{tar_idx + 1}/{len(tar_files)}] Processing {tar_name}")
    print(f"{'='*70}")
    tar_start = time.time()

    try:
        chunks, stats = process_single_tar(tar_path, WORK_DIR)
        all_chunks_combined.extend(chunks)
        # Convert Counter to dict for JSON serialization
        stats["skipped_reasons"] = dict(stats["skipped_reasons"])
        all_stats.append(stats)

        # Mark as completed
        completed_tars.add(tar_name)

        tar_elapsed = time.time() - tar_start
        print(f"    Time: {tar_elapsed:.1f}s")

    except Exception as e:
        print(f"    ERROR processing {tar_name}: {e}")
        all_stats.append({
            "tar_file": tar_name,
            "error": str(e),
        })

    # --- Clean up working directory between tars to save disk ---
    for subdir in ["raw_extracted", "unpacked"]:
        path = os.path.join(WORK_DIR, subdir)
        if os.path.exists(path):
            shutil.rmtree(path)

    # --- Save progress after each tar (resume support) ---
    with open(progress_path, "w") as f:
        json.dump({"completed": list(completed_tars)}, f)

    # Save intermediate combined results
    combined_data = {
        "metadata": {
            "source_dir": TAR_DIR,
            "tar_files_total": len(tar_files),
            "tar_files_processed": len(completed_tars),
            "chunk_word_count_target": CHUNK_WORD_COUNT,
            "chunks_per_paper": CHUNKS_PER_PAPER,
            "min_body_words": MIN_BODY_WORDS,
            "random_seed": RANDOM_SEED,
            "total_chunks": len(all_chunks_combined),
        },
        "per_tar_stats": all_stats,
        "chunks": all_chunks_combined,
    }
    combined_path = os.path.join(OUTPUT_DIR, "all_chunks_combined.json")
    with open(combined_path, "w", encoding="utf-8") as f:
        json.dump(combined_data, f, indent=2, ensure_ascii=False)

    print(f"    Progress saved. Total chunks so far: {len(all_chunks_combined)}")

total_elapsed = time.time() - total_start_time
print(f"\n{'='*70}")
print(f"ALL DONE!")
print(f"{'='*70}")
print(f"Total tar files processed: {len(completed_tars)}/{len(tar_files)}")
print(f"Total chunks: {len(all_chunks_combined)}")
print(f"Total time: {total_elapsed / 60:.1f} minutes")

##4. Saving Final Output

In [ ]:
# =============================================================================
# CELL 4: Save final outputs
# =============================================================================

# Save combined JSON (already saved incrementally, but finalize here)
combined_path = os.path.join(OUTPUT_DIR, "all_chunks_combined.json")
combined_data = {
    "metadata": {
        "source_dir": TAR_DIR,
        "tar_files_total": len(tar_files),
        "tar_files_processed": len(completed_tars),
        "chunk_word_count_target": CHUNK_WORD_COUNT,
        "chunks_per_paper": CHUNKS_PER_PAPER,
        "min_body_words": MIN_BODY_WORDS,
        "random_seed": RANDOM_SEED,
        "total_chunks": len(all_chunks_combined),
    },
    "per_tar_stats": all_stats,
    "chunks": all_chunks_combined,
}
with open(combined_path, "w", encoding="utf-8") as f:
    json.dump(combined_data, f, indent=2, ensure_ascii=False)

# Save combined CSV
csv_path = os.path.join(OUTPUT_DIR, "all_chunks_combined.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["paper_id", "source_tar", "chunk_index", "zone", "word_count", "text"])
    for chunk in all_chunks_combined:
        writer.writerow([
            chunk["paper_id"],
            chunk["source_tar"],
            chunk["chunk_index"],
            chunk["zone"],
            chunk["word_count"],
            chunk["text"],
        ])

print(f"\nFinal outputs saved to {OUTPUT_DIR}:")
print(f"  JSON: {combined_path} ({os.path.getsize(combined_path) / 1024 / 1024:.2f} MB)")
print(f"  CSV:  {csv_path} ({os.path.getsize(csv_path) / 1024 / 1024:.2f} MB)")

# Print summary stats
print(f"\n{'='*70}")
print(f"SUMMARY")
print(f"{'='*70}")

total_papers_in_tars = sum(s.get("total_items", 0) for s in all_stats if "error" not in s)
total_unpacked = sum(s.get("unpacked", 0) for s in all_stats if "error" not in s)
total_tex = sum(s.get("tex_found", 0) for s in all_stats if "error" not in s)
total_processed = sum(s.get("processed", 0) for s in all_stats if "error" not in s)
total_with_chunks = sum(s.get("papers_with_chunks", 0) for s in all_stats if "error" not in s)

print(f"  Papers in tars:       {total_papers_in_tars}")
print(f"  Successfully unpacked:{total_unpacked}")
print(f"  .tex files found:     {total_tex}")
print(f"  Passed text filters:  {total_processed}")
print(f"  Papers with chunks:   {total_with_chunks}")
print(f"  Total chunks:         {len(all_chunks_combined)}")

# Aggregate skip reasons
combined_skips = Counter()
for s in all_stats:
    if "error" not in s and "skipped_reasons" in s:
        combined_skips.update(s["skipped_reasons"])
if combined_skips:
    print(f"\n  Skip reasons across all tars:")
    for reason, count in combined_skips.most_common():
        print(f"    {reason}: {count}")

if all_chunks_combined:
    wc = [c["word_count"] for c in all_chunks_combined]
    print(f"\n  Chunk word counts:")
    print(f"    Min: {min(wc)}, Max: {max(wc)}, Mean: {sum(wc)/len(wc):.1f}")

# Unique papers
unique_papers = set(c["paper_id"] for c in all_chunks_combined)
print(f"\n  Unique papers in final dataset: {len(unique_papers)}")
print(f"  Avg chunks per paper: {len(all_chunks_combined) / max(len(unique_papers), 1):.1f}")

# Per-tar breakdown
print(f"\n  Per-tar breakdown:")
print(f"  {'Tar File':<40} {'Items':>6} {'Processed':>10} {'Chunks':>8}")
print(f"  {'-'*40} {'-'*6} {'-'*10} {'-'*8}")
for s in all_stats:
    if "error" not in s:
        print(f"  {s['tar_file']:<40} {s['total_items']:>6} {s['processed']:>10} {s['total_chunks']:>8}")
    else:
        print(f"  {s['tar_file']:<40} {'ERROR':>6}")


## 5. Quality Inspection

In [ ]:
# =============================================================================
# CELL 5: Quality inspection
# =============================================================================

def inspect_random_chunks(chunks: List[Dict], n: int = 5):
    """Print random chunks for manual quality inspection."""
    sample = random.sample(chunks, min(n, len(chunks)))
    for i, chunk in enumerate(sample):
        print(f"\n{'='*80}")
        print(f"SAMPLE {i+1} | Paper: {chunk['paper_id']} | Tar: {chunk['source_tar']} | Zone: {chunk['zone']} | Words: {chunk['word_count']}")
        print(f"{'='*80}")
        print(chunk["text"][:800])
        if len(chunk["text"]) > 800:
            print("... [truncated for display]")


print("\n\nRANDOM QUALITY INSPECTION:")
if all_chunks_combined:
    inspect_random_chunks(all_chunks_combined, n=5)
else:
    print("No chunks to inspect!")

#Dataset Analysis

In [ ]:
"""
arXiv Dataset Characterization & Analysis
==========================================
Runs comparative analyses on pre-LLM (Dec 2021) and post-LLM (Dec 2025) datasets.
Produces publication-ready visualizations.

Usage:
    1. Set CSV paths in Cell 1
    2. Run all cells
    3. Figures are saved to OUTPUT_DIR and displayed inline

Dependencies (install in Colab):
    !pip install matplotlib seaborn wordcloud nltk textstat pandas numpy
"""

# =============================================================================
# CELL 1: Configuration & Data Loading
# =============================================================================

import os
import re
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from typing import List, Dict

# Install and import additional libraries
# Run these in a separate cell first if needed:
# !pip install wordcloud nltk textstat
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

try:
    import textstat
    HAS_TEXTSTAT = True
except ImportError:
    HAS_TEXTSTAT = False
    print("textstat not installed — readability scores will be skipped")
    print("Install with: !pip install textstat")

# --- CONFIGURATION ---
CSV_2021 = "/content/drive/MyDrive/arxiv/processed/dec_2021/all_chunks_combined.csv"  # <-- CHANGE
CSV_2025 = "/content/drive/MyDrive/arxiv/processed/may_2024/all_chunks_combined.csv"  # <-- CHANGE
OUTPUT_DIR = "/content/drive/MyDrive/arxiv/analysis_figures"
RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- PLOT STYLE ---
# Publication-quality settings
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.figsize': (10, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Color palette — distinct colors for the two periods
C_2021 = "#2E86AB"  # Blue for pre-LLM
C_2025 = "#E84855"  # Red for post-LLM
PALETTE = {"Dec 2021 (Pre-LLM)": C_2021, "Dec 2025 (Post-LLM)": C_2025}

# --- LOAD DATA ---
df_2021 = pd.read_csv(CSV_2021)
df_2025 = pd.read_csv(CSV_2025)

# Add period labels
df_2021["period"] = "Dec 2021 (Pre-LLM)"
df_2025["period"] = "Dec 2025 (Post-LLM)"

# Combine for comparative analysis
df = pd.concat([df_2021, df_2025], ignore_index=True)

print(f"Dec 2021: {len(df_2021)} chunks from {df_2021['paper_id'].nunique()} papers")
print(f"Dec 2025: {len(df_2025)} chunks from {df_2025['paper_id'].nunique()} papers")
print(f"Combined: {len(df)} chunks from {df['paper_id'].nunique()} papers")


# =============================================================================
# CELL 2: Helper functions
# =============================================================================

STOP_WORDS = set(stopwords.words('english'))
# Add domain-specific stopwords common in academic writing
STOP_WORDS.update([
    'also', 'however', 'therefore', 'thus', 'hence', 'moreover',
    'furthermore', 'using', 'used', 'use', 'one', 'two', 'first',
    'second', 'show', 'shown', 'shows', 'result', 'results',
    'figure', 'table', 'section', 'paper', 'work', 'proposed',
    'approach', 'method', 'model', 'data', 'based', 'given',
    'may', 'can', 'e', 'g', 'i', 'et', 'al', 'etc', 'fig',
    'eq', 'ref', 'see', 'note', 'following', 'respectively',
])


def get_words(text: str) -> List[str]:
    """Tokenize and lowercase, keeping only alphabetic tokens."""
    return [w.lower() for w in re.findall(r"[a-zA-Z]+", str(text)) if len(w) > 1]


def get_content_words(text: str) -> List[str]:
    """Get words with stopwords removed."""
    return [w for w in get_words(text) if w not in STOP_WORDS]


def get_sentences(text: str) -> List[str]:
    """Split text into sentences."""
    return sent_tokenize(str(text))


def type_token_ratio(words: List[str]) -> float:
    """Calculate type-token ratio (lexical diversity)."""
    if not words:
        return 0.0
    return len(set(words)) / len(words)


def get_ngrams(words: List[str], n: int) -> List[str]:
    """Get n-grams from a word list."""
    return [" ".join(words[i:i+n]) for i in range(len(words) - n + 1)]


def save_fig(fig, filename: str):
    """Save figure to output directory."""
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, bbox_inches='tight', facecolor='white')
    print(f"  Saved: {path}")


# Pre-compute text features for each chunk
print("Computing text features (this may take a minute)...")

def compute_features(row):
    text = str(row['text'])
    words = get_words(text)
    content_words = get_content_words(text)
    sentences = get_sentences(text)
    sent_lengths = [len(s.split()) for s in sentences]

    features = {
        'n_words': len(words),
        'n_unique_words': len(set(words)),
        'n_sentences': len(sentences),
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'avg_sentence_length': np.mean(sent_lengths) if sent_lengths else 0,
        'std_sentence_length': np.std(sent_lengths) if len(sent_lengths) > 1 else 0,
        'type_token_ratio': type_token_ratio(words),
        'content_type_token_ratio': type_token_ratio(content_words),
        'long_word_ratio': sum(1 for w in words if len(w) > 8) / max(len(words), 1),
        'short_sentence_ratio': sum(1 for l in sent_lengths if l < 10) / max(len(sent_lengths), 1),
        'long_sentence_ratio': sum(1 for l in sent_lengths if l > 35) / max(len(sent_lengths), 1),
    }

    if HAS_TEXTSTAT:
        features['flesch_reading_ease'] = textstat.flesch_reading_ease(text)
        features['flesch_kincaid_grade'] = textstat.flesch_kincaid_grade(text)
        features['coleman_liau_index'] = textstat.coleman_liau_index(text)

    return pd.Series(features)

features_df = df.apply(compute_features, axis=1)
df = pd.concat([df, features_df], axis=1)

print("Done computing features.")
print(f"\nFeature columns added: {list(features_df.columns)}")


# =============================================================================
# CELL 3: Analysis 1 — Dataset Overview & Basic Statistics
# =============================================================================

print("=" * 70)
print("ANALYSIS 1: Basic Dataset Statistics")
print("=" * 70)

# Summary statistics table
stats_cols = [
    'n_words', 'n_sentences', 'avg_word_length', 'avg_sentence_length',
    'type_token_ratio', 'content_type_token_ratio', 'long_word_ratio',
]
if HAS_TEXTSTAT:
    stats_cols += ['flesch_reading_ease', 'flesch_kincaid_grade', 'coleman_liau_index']

summary = df.groupby('period')[stats_cols].agg(['mean', 'std', 'median']).round(3)
print("\nSummary Statistics by Period:")
print(summary.to_string())

# Create a clean comparison figure
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Dataset Characterization: Dec 2021 vs Dec 2025", fontsize=16, fontweight='bold', y=1.02)

plot_features = [
    ('n_words', 'Word Count per Chunk'),
    ('avg_sentence_length', 'Avg Sentence Length (words)'),
    ('type_token_ratio', 'Type-Token Ratio (Lexical Diversity)'),
    ('avg_word_length', 'Average Word Length (chars)'),
    ('long_word_ratio', 'Proportion of Long Words (>8 chars)'),
    ('std_sentence_length', 'Sentence Length Std Dev'),
]

for idx, (col, title) in enumerate(plot_features):
    ax = axes[idx // 3][idx % 3]
    for period, color in PALETTE.items():
        data = df[df['period'] == period][col].dropna()
        ax.hist(data, bins=30, alpha=0.6, color=color, label=period, density=True, edgecolor='white', linewidth=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Density')
    if idx == 0:
        ax.legend(frameon=False)

plt.tight_layout()
save_fig(fig, "01_basic_statistics.png")
plt.show()

# Print the key comparison
print("\n\nKey Metrics Comparison:")
print(f"{'Metric':<35} {'Dec 2021':>12} {'Dec 2025':>12} {'Difference':>12}")
print("-" * 75)
for col in stats_cols:
    m21 = df[df['period'] == 'Dec 2021 (Pre-LLM)'][col].mean()
    m25 = df[df['period'] == 'Dec 2025 (Post-LLM)'][col].mean()
    diff = m25 - m21
    pct = (diff / m21 * 100) if m21 != 0 else 0
    print(f"{col:<35} {m21:>12.3f} {m25:>12.3f} {diff:>+8.3f} ({pct:>+.1f}%)")


# =============================================================================
# CELL 4: Analysis 2 — Word Clouds
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSIS 2: Word Clouds")
print("=" * 70)

def make_word_freq(texts: pd.Series) -> Counter:
    """Build word frequency counter from a series of texts."""
    freq = Counter()
    for text in texts:
        freq.update(get_content_words(str(text)))
    return freq

freq_2021 = make_word_freq(df_2021['text'])
freq_2025 = make_word_freq(df_2025['text'])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Most Frequent Content Words", fontsize=16, fontweight='bold')

# Colormap functions for each period
def blue_color_func(*args, **kwargs):
    return f"hsl(200, {random.randint(50, 80)}%, {random.randint(25, 50)}%)"

def red_color_func(*args, **kwargs):
    return f"hsl(355, {random.randint(50, 80)}%, {random.randint(30, 55)}%)"

wc_2021 = WordCloud(
    width=800, height=500, background_color='white',
    max_words=100, color_func=blue_color_func, random_state=RANDOM_SEED,
    prefer_horizontal=0.7, min_font_size=10,
).generate_from_frequencies(freq_2021)

wc_2025 = WordCloud(
    width=800, height=500, background_color='white',
    max_words=100, color_func=red_color_func, random_state=RANDOM_SEED,
    prefer_horizontal=0.7, min_font_size=10,
).generate_from_frequencies(freq_2025)

axes[0].imshow(wc_2021, interpolation='bilinear')
axes[0].set_title("Dec 2021 (Pre-LLM)", fontsize=14, fontweight='bold', color=C_2021)
axes[0].axis('off')

axes[1].imshow(wc_2025, interpolation='bilinear')
axes[1].set_title("Dec 2025 (Post-LLM)", fontsize=14, fontweight='bold', color=C_2025)
axes[1].axis('off')

plt.tight_layout()
save_fig(fig, "02_word_clouds.png")
plt.show()


# =============================================================================
# CELL 5: Analysis 3 — Distinctive Vocabulary (Log-Odds Ratio)
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSIS 3: Distinctive Vocabulary Between Periods")
print("=" * 70)

def log_odds_ratio(freq_a: Counter, freq_b: Counter, min_count: int = 20) -> pd.DataFrame:
    """
    Compute log-odds ratio with informative Dirichlet prior.
    Identifies words disproportionately associated with each corpus.
    """
    total_a = sum(freq_a.values())
    total_b = sum(freq_b.values())
    all_words = set(freq_a.keys()) | set(freq_b.keys())

    # Filter by minimum count
    all_words = {w for w in all_words if freq_a[w] + freq_b[w] >= min_count}

    rows = []
    alpha = 1  # Dirichlet prior smoothing

    for word in all_words:
        n_a = freq_a[word] + alpha
        n_b = freq_b[word] + alpha
        rate_a = n_a / (total_a + alpha * len(all_words))
        rate_b = n_b / (total_b + alpha * len(all_words))

        log_odds = np.log(rate_b / rate_a)
        # Approximate variance for confidence
        variance = 1/n_a + 1/n_b
        z_score = log_odds / np.sqrt(variance)

        rows.append({
            'word': word,
            'count_2021': freq_a[word],
            'count_2025': freq_b[word],
            'log_odds': log_odds,
            'z_score': z_score,
            'total_count': freq_a[word] + freq_b[word],
        })

    return pd.DataFrame(rows).sort_values('z_score', ascending=False)

lor_df = log_odds_ratio(freq_2021, freq_2025)

# Top distinctive words for each period
n_show = 20
top_2025 = lor_df.head(n_show)
top_2021 = lor_df.tail(n_show).iloc[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("Most Distinctive Words by Period (Log-Odds Ratio)", fontsize=16, fontweight='bold')

# Words more associated with 2025
axes[0].barh(
    top_2025['word'], top_2025['z_score'],
    color=C_2025, edgecolor='white', linewidth=0.5
)
axes[0].set_title(f"More frequent in Dec 2025", fontweight='bold', color=C_2025)
axes[0].set_xlabel("Z-score (log-odds ratio)")
axes[0].invert_yaxis()

# Words more associated with 2021
axes[1].barh(
    top_2021['word'], -top_2021['z_score'],
    color=C_2021, edgecolor='white', linewidth=0.5
)
axes[1].set_title(f"More frequent in Dec 2021", fontweight='bold', color=C_2021)
axes[1].set_xlabel("Z-score (log-odds ratio)")
axes[1].invert_yaxis()

plt.tight_layout()
save_fig(fig, "03_distinctive_vocabulary.png")
plt.show()

print("\nTop 20 words MORE frequent in Dec 2025:")
for _, row in top_2025.iterrows():
    print(f"  {row['word']:<20} z={row['z_score']:>6.2f}  (2021: {row['count_2021']:>5}, 2025: {row['count_2025']:>5})")

print("\nTop 20 words MORE frequent in Dec 2021:")
for _, row in top_2021.iterrows():
    print(f"  {row['word']:<20} z={row['z_score']:>6.2f}  (2021: {row['count_2021']:>5}, 2025: {row['count_2025']:>5})")


# =============================================================================
# CELL 6: Analysis 4 — N-gram Analysis (LLM Phrase Detection)
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSIS 4: N-gram Analysis & Formulaic Phrases")
print("=" * 70)

def get_ngram_freq(texts: pd.Series, n: int, min_count: int = 5) -> Counter:
    """Get n-gram frequencies from a text corpus."""
    freq = Counter()
    for text in texts:
        words = get_words(str(text))
        freq.update(get_ngrams(words, n))
    return Counter({k: v for k, v in freq.items() if v >= min_count})


# Compute bigram and trigram frequencies
bigrams_2021 = get_ngram_freq(df_2021['text'], 2)
bigrams_2025 = get_ngram_freq(df_2025['text'], 2)
trigrams_2021 = get_ngram_freq(df_2021['text'], 3)
trigrams_2025 = get_ngram_freq(df_2025['text'], 3)

# Known LLM-associated phrases (from prior research)
llm_phrases = [
    "it is worth noting", "plays a crucial role", "in recent years",
    "it is important to note", "a comprehensive overview",
    "delve into", "in the realm of", "shed light on",
    "a myriad of", "in this context", "notably",
    "leveraging the power", "a pivotal role", "offers a promising",
    "a comprehensive understanding", "it is noteworthy",
    "groundbreaking", "a nuanced understanding", "the intricacies of",
    "multifaceted", "the landscape of",
]

# Check LLM-phrase frequency
print("\nKnown LLM-associated phrases — occurrence counts:")
print(f"{'Phrase':<35} {'Dec 2021':>10} {'Dec 2025':>10} {'Ratio':>10}")
print("-" * 70)

# Normalize by total chunks
n_2021 = len(df_2021)
n_2025 = len(df_2025)

phrase_data = []
for phrase in llm_phrases:
    count_2021 = sum(1 for t in df_2021['text'] if phrase.lower() in str(t).lower())
    count_2025 = sum(1 for t in df_2025['text'] if phrase.lower() in str(t).lower())
    rate_2021 = count_2021 / n_2021 * 1000  # per 1000 chunks
    rate_2025 = count_2025 / n_2025 * 1000
    ratio = rate_2025 / rate_2021 if rate_2021 > 0 else float('inf')

    phrase_data.append({
        'phrase': phrase, 'count_2021': count_2021, 'count_2025': count_2025,
        'rate_2021': rate_2021, 'rate_2025': rate_2025, 'ratio': ratio,
    })
    if count_2021 > 0 or count_2025 > 0:
        ratio_str = f"{ratio:.1f}x" if ratio != float('inf') else "∞"
        print(f"{phrase:<35} {count_2021:>10} {count_2025:>10} {ratio_str:>10}")

phrase_df = pd.DataFrame(phrase_data)
phrase_df = phrase_df[(phrase_df['count_2021'] > 0) | (phrase_df['count_2025'] > 0)]
phrase_df = phrase_df.sort_values('ratio', ascending=False)

# Plot LLM phrase comparison
if len(phrase_df) > 0:
    fig, ax = plt.subplots(figsize=(14, max(6, len(phrase_df) * 0.4)))
    fig.suptitle("LLM-Associated Phrases: Rate per 1000 Chunks", fontsize=16, fontweight='bold')

    y_pos = range(len(phrase_df))
    bar_height = 0.35

    ax.barh(
        [y - bar_height/2 for y in y_pos],
        phrase_df['rate_2021'], bar_height,
        color=C_2021, label='Dec 2021', edgecolor='white', linewidth=0.5
    )
    ax.barh(
        [y + bar_height/2 for y in y_pos],
        phrase_df['rate_2025'], bar_height,
        color=C_2025, label='Dec 2025', edgecolor='white', linewidth=0.5
    )

    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(phrase_df['phrase'])
    ax.set_xlabel('Occurrences per 1000 chunks')
    ax.legend(frameon=False)
    ax.invert_yaxis()

    plt.tight_layout()
    save_fig(fig, "04_llm_phrases.png")
    plt.show()

# Top trigrams comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("Top 20 Trigrams by Period", fontsize=16, fontweight='bold')

for idx, (freq, period, color) in enumerate([
    (trigrams_2021, "Dec 2021", C_2021),
    (trigrams_2025, "Dec 2025", C_2025),
]):
    top = freq.most_common(20)
    if top:
        words, counts = zip(*top)
        axes[idx].barh(words, counts, color=color, edgecolor='white', linewidth=0.5)
        axes[idx].set_title(period, fontweight='bold', color=color)
        axes[idx].set_xlabel('Frequency')
        axes[idx].invert_yaxis()

plt.tight_layout()
save_fig(fig, "04b_top_trigrams.png")
plt.show()


# =============================================================================
# CELL 7: Analysis 5 — Readability & Sentence Structure
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSIS 5: Readability & Sentence Structure")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Sentence Structure & Readability Comparison", fontsize=16, fontweight='bold', y=1.02)

# 5a: Sentence length distribution
ax = axes[0][0]
for period, color in PALETTE.items():
    data = df[df['period'] == period]['avg_sentence_length'].dropna()
    ax.hist(data, bins=30, alpha=0.6, color=color, label=period, density=True, edgecolor='white', linewidth=0.5)
ax.set_title("Average Sentence Length Distribution", fontweight='bold')
ax.set_xlabel("Words per sentence")
ax.set_ylabel("Density")
ax.legend(frameon=False)

# 5b: Sentence length variability (uniformity check)
ax = axes[0][1]
for period, color in PALETTE.items():
    data = df[df['period'] == period]['std_sentence_length'].dropna()
    ax.hist(data, bins=30, alpha=0.6, color=color, label=period, density=True, edgecolor='white', linewidth=0.5)
ax.set_title("Sentence Length Variability (Std Dev)", fontweight='bold')
ax.set_xlabel("Std dev of sentence lengths within chunk")
ax.set_ylabel("Density")
ax.legend(frameon=False)

# 5c: Readability scores
if HAS_TEXTSTAT:
    ax = axes[1][0]
    for period, color in PALETTE.items():
        data = df[df['period'] == period]['flesch_kincaid_grade'].dropna()
        # Clip extreme values
        data = data.clip(-5, 30)
        ax.hist(data, bins=30, alpha=0.6, color=color, label=period, density=True, edgecolor='white', linewidth=0.5)
    ax.set_title("Flesch-Kincaid Grade Level", fontweight='bold')
    ax.set_xlabel("Grade level")
    ax.set_ylabel("Density")
    ax.legend(frameon=False)
else:
    axes[1][0].text(0.5, 0.5, "textstat not installed", ha='center', va='center', transform=axes[1][0].transAxes)

# 5d: Short vs long sentence ratios
ax = axes[1][1]
metrics = ['short_sentence_ratio', 'long_sentence_ratio']
x = np.arange(len(metrics))
width = 0.35

for i, (period, color) in enumerate(PALETTE.items()):
    means = [df[df['period'] == period][m].mean() for m in metrics]
    stds = [df[df['period'] == period][m].std() for m in metrics]
    ax.bar(x + i * width, means, width, yerr=stds, color=color, label=period,
           edgecolor='white', linewidth=0.5, capsize=3, alpha=0.8)

ax.set_title("Sentence Length Extremes", fontweight='bold')
ax.set_xticks(x + width / 2)
ax.set_xticklabels(["Short sentences\n(<10 words)", "Long sentences\n(>35 words)"])
ax.set_ylabel("Proportion of sentences")
ax.legend(frameon=False)

plt.tight_layout()
save_fig(fig, "05_readability_sentence_structure.png")
plt.show()

# Print readability comparison
if HAS_TEXTSTAT:
    print("\nReadability Scores Comparison:")
    for metric in ['flesch_reading_ease', 'flesch_kincaid_grade', 'coleman_liau_index']:
        m21 = df[df['period'] == 'Dec 2021 (Pre-LLM)'][metric].mean()
        m25 = df[df['period'] == 'Dec 2025 (Post-LLM)'][metric].mean()
        print(f"  {metric:<30} 2021: {m21:.2f}  |  2025: {m25:.2f}  |  Δ: {m25-m21:+.2f}")


# =============================================================================
# CELL 8: Analysis 6 — Lexical Diversity by Zone (Position in Paper)
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSIS 6: Writing Characteristics by Position in Paper")
print("=" * 70)

zone_features = ['type_token_ratio', 'avg_sentence_length', 'avg_word_length']
zone_labels = {0: 'Intro', 1: 'Early-Mid', 2: 'Late-Mid', 3: 'Conclusion'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Writing Style by Position in Paper", fontsize=16, fontweight='bold')

for idx, feat in enumerate(zone_features):
    ax = axes[idx]

    # Compute means by zone and period
    for period, color in PALETTE.items():
        subset = df[df['period'] == period]
        zone_means = subset.groupby('zone')[feat].mean()
        zone_stds = subset.groupby('zone')[feat].std()
        zones = sorted(zone_means.index)
        labels = [zone_labels.get(z, f'Zone {z}') for z in zones]

        ax.plot(labels, [zone_means[z] for z in zones], 'o-', color=color, label=period, linewidth=2, markersize=6)
        ax.fill_between(
            labels,
            [zone_means[z] - zone_stds[z] for z in zones],
            [zone_means[z] + zone_stds[z] for z in zones],
            alpha=0.15, color=color
        )

    nice_names = {
        'type_token_ratio': 'Lexical Diversity (TTR)',
        'avg_sentence_length': 'Avg Sentence Length',
        'avg_word_length': 'Avg Word Length',
    }
    ax.set_title(nice_names.get(feat, feat), fontweight='bold')
    ax.set_ylabel(feat.replace('_', ' ').title())
    if idx == 0:
        ax.legend(frameon=False)

plt.tight_layout()
save_fig(fig, "06_by_paper_position.png")
plt.show()


# =============================================================================
# CELL 9: Analysis 7 — Punctuation & Character Patterns
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSIS 7: Punctuation & Character Patterns")
print("=" * 70)

def punctuation_features(text: str) -> Dict[str, float]:
    """Compute punctuation frequencies normalized by word count."""
    text = str(text)
    words = text.split()
    n = max(len(words), 1)
    return {
        'semicolons': text.count(';') / n * 100,
        'colons': text.count(':') / n * 100,
        'exclamations': text.count('!') / n * 100,
        'questions': text.count('?') / n * 100,
        'parentheses': (text.count('(') + text.count(')')) / n * 100,
        'dashes': (text.count('—') + text.count('–') + text.count(' - ')) / n * 100,
        'commas': text.count(',') / n * 100,
        'quotes': (text.count('"') + text.count("'") + text.count('"') + text.count('"')) / n * 100,
    }

punct_rows = df['text'].apply(lambda t: pd.Series(punctuation_features(t)))
df_punct = pd.concat([df[['period']], punct_rows], axis=1)

fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle("Punctuation Frequency per 100 Words", fontsize=16, fontweight='bold')

punct_cols = ['commas', 'parentheses', 'semicolons', 'colons', 'dashes', 'quotes', 'questions', 'exclamations']
x = np.arange(len(punct_cols))
width = 0.35

for i, (period, color) in enumerate(PALETTE.items()):
    means = [df_punct[df_punct['period'] == period][c].mean() for c in punct_cols]
    stds = [df_punct[df_punct['period'] == period][c].std() for c in punct_cols]
    ax.bar(x + i * width, means, width, yerr=stds, color=color, label=period,
           edgecolor='white', linewidth=0.5, capsize=3, alpha=0.8)

ax.set_xticks(x + width / 2)
ax.set_xticklabels([c.title() for c in punct_cols], rotation=15, ha='right')
ax.set_ylabel("Count per 100 words")
ax.legend(frameon=False)

plt.tight_layout()
save_fig(fig, "07_punctuation_patterns.png")
plt.show()

# Print comparison
print("\nPunctuation per 100 words:")
print(f"{'Mark':<20} {'Dec 2021':>10} {'Dec 2025':>10} {'Change':>10}")
print("-" * 55)
for c in punct_cols:
    m21 = df_punct[df_punct['period'] == 'Dec 2021 (Pre-LLM)'][c].mean()
    m25 = df_punct[df_punct['period'] == 'Dec 2025 (Post-LLM)'][c].mean()
    pct = (m25 - m21) / m21 * 100 if m21 > 0 else 0
    print(f"{c:<20} {m21:>10.3f} {m25:>10.3f} {pct:>+8.1f}%")


# =============================================================================
# CELL 10: Analysis 8 — LaTeX Artifact Check (Data Quality)
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSIS 8: Data Quality — LaTeX Artifact Check")
print("=" * 70)

# Check for common leftover LaTeX artifacts
artifacts = {
    'backslash_commands': r'\\[a-zA-Z]+',
    'curly_braces': r'[{}]',
    'dollar_signs': r'\$',
    'backslash_special': r'\\[&%#_~^]',
    'ampersand': r'&',
    'double_backslash': r'\\\\',
}

print("\nChunks containing LaTeX artifacts:")
print(f"{'Artifact':<25} {'Dec 2021':>12} {'Dec 2021 %':>10} {'Dec 2025':>12} {'Dec 2025 %':>10}")
print("-" * 75)

for name, pattern in artifacts.items():
    count_21 = sum(1 for t in df_2021['text'] if re.search(pattern, str(t)))
    count_25 = sum(1 for t in df_2025['text'] if re.search(pattern, str(t)))
    pct_21 = count_21 / len(df_2021) * 100
    pct_25 = count_25 / len(df_2025) * 100
    print(f"{name:<25} {count_21:>12} {pct_21:>9.1f}% {count_25:>12} {pct_25:>9.1f}%")

# Show a few examples of chunks with artifacts for manual review
print("\n\nSample chunks with backslash commands (potential cleaning issues):")
artifact_chunks = df[df['text'].str.contains(r'\\[a-zA-Z]+', regex=True, na=False)].sample(
    min(3, len(df[df['text'].str.contains(r'\\[a-zA-Z]+', regex=True, na=False)])),
    random_state=RANDOM_SEED
)
for _, row in artifact_chunks.iterrows():
    # Find and highlight the artifact
    matches = re.findall(r'\\[a-zA-Z]+', str(row['text']))
    print(f"\n  Paper: {row['paper_id']} | Artifacts found: {matches[:5]}")
    print(f"  Text preview: {str(row['text'])[:200]}...")


# =============================================================================
# CELL 11: Summary Dashboard
# =============================================================================

print("\n" + "=" * 70)
print("GENERATING SUMMARY DASHBOARD")
print("=" * 70)

fig = plt.figure(figsize=(20, 12))
gs = gridspec.GridSpec(3, 4, hspace=0.4, wspace=0.35)
fig.suptitle("arXiv Dataset Characterization Dashboard\nDec 2021 (Pre-LLM) vs Dec 2025 (Post-LLM)",
             fontsize=18, fontweight='bold', y=0.98)

# 1: Dataset sizes
ax1 = fig.add_subplot(gs[0, 0])
papers = [df_2021['paper_id'].nunique(), df_2025['paper_id'].nunique()]
chunks = [len(df_2021), len(df_2025)]
x = [0, 1]
bars = ax1.bar(x, chunks, color=[C_2021, C_2025], edgecolor='white', width=0.6)
ax1.set_xticks(x)
ax1.set_xticklabels(['2021', '2025'])
ax1.set_title('Total Chunks', fontweight='bold')
for bar, val, p in zip(bars, chunks, papers):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{val}\n({p} papers)', ha='center', va='bottom', fontsize=9)

# 2: Avg sentence length
ax2 = fig.add_subplot(gs[0, 1])
for period, color in PALETTE.items():
    data = df[df['period'] == period]['avg_sentence_length'].dropna()
    ax2.hist(data, bins=25, alpha=0.6, color=color, density=True, edgecolor='white', linewidth=0.3)
ax2.set_title('Sentence Length', fontweight='bold')
ax2.set_xlabel('Words/sentence')

# 3: Lexical diversity
ax3 = fig.add_subplot(gs[0, 2])
for period, color in PALETTE.items():
    data = df[df['period'] == period]['type_token_ratio'].dropna()
    ax3.hist(data, bins=25, alpha=0.6, color=color, density=True, edgecolor='white', linewidth=0.3)
ax3.set_title('Lexical Diversity (TTR)', fontweight='bold')
ax3.set_xlabel('Type-token ratio')

# 4: Readability
ax4 = fig.add_subplot(gs[0, 3])
if HAS_TEXTSTAT:
    for period, color in PALETTE.items():
        data = df[df['period'] == period]['flesch_kincaid_grade'].dropna().clip(-5, 25)
        ax4.hist(data, bins=25, alpha=0.6, color=color, density=True, edgecolor='white', linewidth=0.3)
    ax4.set_title('FK Grade Level', fontweight='bold')
    ax4.set_xlabel('Grade level')
else:
    ax4.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('FK Grade Level', fontweight='bold')

# 5-6: Word clouds
ax5 = fig.add_subplot(gs[1, 0:2])
ax5.imshow(wc_2021, interpolation='bilinear')
ax5.set_title('Top Words — 2021', fontweight='bold', color=C_2021)
ax5.axis('off')

ax6 = fig.add_subplot(gs[1, 2:4])
ax6.imshow(wc_2025, interpolation='bilinear')
ax6.set_title('Top Words — 2025', fontweight='bold', color=C_2025)
ax6.axis('off')

# 7: Distinctive vocabulary (top 10 each)
ax7 = fig.add_subplot(gs[2, 0:2])
top10_2025 = lor_df.head(10)
ax7.barh(top10_2025['word'], top10_2025['z_score'], color=C_2025, edgecolor='white', linewidth=0.5)
ax7.set_title('Most Distinctive — 2025', fontweight='bold', color=C_2025)
ax7.invert_yaxis()
ax7.set_xlabel('Z-score')

ax8 = fig.add_subplot(gs[2, 2:4])
top10_2021 = lor_df.tail(10).iloc[::-1]
ax8.barh(top10_2021['word'], -top10_2021['z_score'], color=C_2021, edgecolor='white', linewidth=0.5)
ax8.set_title('Most Distinctive — 2021', fontweight='bold', color=C_2021)
ax8.invert_yaxis()
ax8.set_xlabel('Z-score')

save_fig(fig, "08_summary_dashboard.png")
plt.show()

print(f"\nAll figures saved to: {OUTPUT_DIR}")
print(f"Files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.png'):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
        print(f"  {f} ({size:.0f} KB)")

# Binoculars Classification

In [ ]:
%cd /content/Binoculars
!pip install --no-deps -e . --quiet
!pip install transformers==4.38.0 tokenizers datasets numpy scikit-learn --quiet

from binoculars import Binoculars
print("Success!")

In [ ]:
# Cell 0a: Install Binoculars
!git clone https://github.com/ahans30/Binoculars.git
%cd /content/Binoculars
!pip install -e .
%cd /content

In [ ]:
# Cell 0b: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
Binoculars Inference Pipeline
==============================
Feeds preprocessed chunks from CSV to Binoculars and stores scores.
Designed for Google Colab with GPU.

Setup (run these in separate cells first):
    !git clone https://github.com/ahans30/Binoculars.git
    %cd /content/Binoculars
    !pip install -e .
    %cd /content

Dependencies:
    - Binoculars (installed above)
    - pandas
    - GPU runtime enabled (Runtime → Change runtime type → GPU)
"""

# =============================================================================
# CELL 1: Configuration & Setup
# =============================================================================

import sys
sys.path.insert(0, "/content/Binoculars")

import os
import time
import json
import torch
import pandas as pd
import numpy as np
from datetime import datetime

# --- CONFIGURATION ---
# Input CSV files (from preprocessing pipeline)
CSV_2021 = "/content/drive/MyDrive/arxiv/processed/dec_2021/all_chunks_combined.csv"  # <-- CHANGE
CSV_2025 = "/content/drive/MyDrive/arxiv/processed/may_2024/all_chunks_combined.csv"  # <-- CHANGE

# Output directory for results
OUTPUT_DIR = "/content/drive/MyDrive/arxiv/binoculars_results"

# Binoculars settings
BATCH_SIZE = 8             # Chunks per batch — lower this if you get OOM errors
USE_FP16 = True            # Use half precision to save VRAM

# HuggingFace cache on Drive (avoids re-downloading models on session restart)
HF_CACHE = "/content/drive/MyDrive/arxiv/hf_cache"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE

# Verify GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! This will be extremely slow on CPU.")

print(f"\nBatch size: {BATCH_SIZE}")
print(f"FP16 mode: {USE_FP16}")
print(f"HF cache: {HF_CACHE}")


# =============================================================================
# CELL 2: Load Binoculars model
# =============================================================================

# Force FP16 if configured (must be set before importing Binoculars)
if USE_FP16:
    torch.set_default_dtype(torch.float16)
    print("Set default dtype to float16")

from binoculars import Binoculars

print("Loading Binoculars models (Falcon-7B + Falcon-7B-Instruct)...")
print("This may take a few minutes on first run (downloading ~14GB of models)...")
load_start = time.time()

bino = Binoculars()

load_time = time.time() - load_start
print(f"Models loaded in {load_time:.1f}s")

# Reset default dtype so pandas/numpy operations aren't affected
if USE_FP16:
    torch.set_default_dtype(torch.float32)

# Quick sanity check
test_score = bino.compute_score("This is a simple test sentence to verify the model is working.")
test_pred = bino.predict("This is a simple test sentence to verify the model is working.")
print(f"Sanity check — score: {test_score:.4f}, prediction: {test_pred}")

# Check VRAM usage after loading
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"VRAM used: {allocated:.1f} GB allocated, {reserved:.1f} GB reserved")


# =============================================================================
# CELL 3: Define scoring function with batching and checkpointing
# =============================================================================

def score_dataset(
    csv_path: str,
    output_dir: str,
    bino_model,
    batch_size: int = 8,
    dataset_label: str = "dataset",
):
    """
    Score all chunks in a CSV file using Binoculars.

    Features:
    - Batched inference for efficiency
    - Checkpointing after each batch (resume on crash)
    - Progress logging
    - Saves both raw scores and binary predictions

    Returns: DataFrame with scores added
    """

    # --- Load data ---
    df = pd.read_csv(csv_path)
    total_chunks = len(df)
    print(f"\nScoring {dataset_label}: {total_chunks} chunks from {csv_path}")

    # --- Check for existing checkpoint ---
    checkpoint_path = os.path.join(output_dir, f"checkpoint_{dataset_label}.json")
    start_idx = 0
    scores = [None] * total_chunks
    predictions = [None] * total_chunks

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            checkpoint = json.load(f)
        start_idx = checkpoint["next_idx"]
        scores = checkpoint["scores"]
        predictions = checkpoint["predictions"]
        print(f"  Resuming from checkpoint: {start_idx}/{total_chunks} already done")

    # --- Batch scoring loop ---
    total_batches = (total_chunks - start_idx + batch_size - 1) // batch_size
    batch_times = []

    for batch_num, i in enumerate(range(start_idx, total_chunks, batch_size)):
        batch_start = time.time()

        # Get batch of texts
        batch_end = min(i + batch_size, total_chunks)
        batch_texts = df["text"].iloc[i:batch_end].tolist()

        # Ensure all items are strings (handle NaN or non-string)
        batch_texts = [str(t) if pd.notna(t) else "" for t in batch_texts]

        # Skip empty texts
        valid_mask = [len(t.strip()) > 0 for t in batch_texts]

        try:
            if any(valid_mask):
                valid_texts = [t for t, m in zip(batch_texts, valid_mask) if m]

                # Compute scores — Binoculars accepts a list of strings
                batch_scores = bino.compute_score(valid_texts)
                batch_preds = bino.predict(valid_texts)

                # Handle single item (returns scalar instead of list)
                if isinstance(batch_scores, (int, float)):
                    batch_scores = [batch_scores]
                    batch_preds = [batch_preds]

                # Map back to full batch (including invalid items)
                valid_idx = 0
                for j, is_valid in enumerate(valid_mask):
                    if is_valid:
                        scores[i + j] = float(batch_scores[valid_idx])
                        predictions[i + j] = str(batch_preds[valid_idx])
                        valid_idx += 1
                    else:
                        scores[i + j] = None
                        predictions[i + j] = "skipped_empty"

        except Exception as e:
            print(f"\n  ERROR in batch {batch_num}: {e}")
            # Score individually as fallback
            for j in range(len(batch_texts)):
                try:
                    if valid_mask[j]:
                        s = bino.compute_score(batch_texts[j])
                        p = bino.predict(batch_texts[j])
                        scores[i + j] = float(s)
                        predictions[i + j] = str(p)
                    else:
                        scores[i + j] = None
                        predictions[i + j] = "skipped_empty"
                except Exception as inner_e:
                    scores[i + j] = None
                    predictions[i + j] = f"error: {str(inner_e)[:50]}"

        batch_elapsed = time.time() - batch_start
        batch_times.append(batch_elapsed)

        # --- Save checkpoint ---
        checkpoint_data = {
            "next_idx": batch_end,
            "scores": scores,
            "predictions": predictions,
            "dataset_label": dataset_label,
            "timestamp": datetime.now().isoformat(),
        }
        with open(checkpoint_path, "w") as f:
            json.dump(checkpoint_data, f)

        # --- Progress logging ---
        completed = batch_end - start_idx
        total_remaining = total_chunks - batch_end
        avg_time = np.mean(batch_times[-10:])  # Moving average of last 10 batches
        eta_seconds = (total_remaining / batch_size) * avg_time
        eta_minutes = eta_seconds / 60

        chunks_per_sec = batch_size / avg_time if avg_time > 0 else 0

        # Print progress every 10 batches or on first/last
        if batch_num % 10 == 0 or batch_end >= total_chunks:
            print(
                f"  [{batch_end:>6}/{total_chunks}] "
                f"({batch_end/total_chunks*100:5.1f}%) "
                f"| {chunks_per_sec:.1f} chunks/s "
                f"| batch: {batch_elapsed:.2f}s "
                f"| ETA: {eta_minutes:.1f} min"
            )

    # --- Add scores to dataframe ---
    df["binoculars_score"] = scores
    df["binoculars_prediction"] = predictions

    # --- Save results ---
    output_csv = os.path.join(output_dir, f"scored_{dataset_label}.csv")
    df.to_csv(output_csv, index=False)
    print(f"\n  Results saved to: {output_csv}")

    # --- Clean up checkpoint (processing complete) ---
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        print(f"  Checkpoint cleaned up")

    # --- Print summary ---
    valid_scores = [s for s in scores if s is not None]
    if valid_scores:
        print(f"\n  Score Summary ({dataset_label}):")
        print(f"    Scored: {len(valid_scores)}/{total_chunks} chunks")
        print(f"    Mean score:   {np.mean(valid_scores):.4f}")
        print(f"    Median score: {np.median(valid_scores):.4f}")
        print(f"    Std dev:      {np.std(valid_scores):.4f}")
        print(f"    Min:          {np.min(valid_scores):.4f}")
        print(f"    Max:          {np.max(valid_scores):.4f}")

        # Count predictions
        pred_counts = pd.Series(predictions).value_counts()
        print(f"\n    Predictions:")
        for pred, count in pred_counts.items():
            if pred is not None:
                pct = count / total_chunks * 100
                print(f"      {pred}: {count} ({pct:.1f}%)")

    total_time = sum(batch_times)
    print(f"\n    Total scoring time: {total_time/60:.1f} minutes")

    return df


# =============================================================================
# CELL 4: Score December 2021 dataset (pre-LLM)
# =============================================================================

print("=" * 70)
print("SCORING: December 2021 (Pre-LLM)")
print("=" * 70)

df_2021_scored = score_dataset(
    csv_path=CSV_2021,
    output_dir=OUTPUT_DIR,
    bino_model=bino,
    batch_size=BATCH_SIZE,
    dataset_label="dec_2021",
)


# =============================================================================
# CELL 5: Score December 2025 dataset (post-LLM)
# =============================================================================

print("\n" + "=" * 70)
print("SCORING: December 2025 (Post-LLM)")
print("=" * 70)

df_2025_scored = score_dataset(
    csv_path=CSV_2025,
    output_dir=OUTPUT_DIR,
    bino_model=bino,
    batch_size=BATCH_SIZE,
    dataset_label="may_2024",
)


# =============================================================================
# CELL 6: Comparative summary
# =============================================================================

print("\n" + "=" * 70)
print("COMPARATIVE SUMMARY")
print("=" * 70)

def summarize_scores(df, label):
    """Compute summary statistics for a scored dataset."""
    valid = df["binoculars_score"].dropna()
    ai_pred = (df["binoculars_prediction"] == "Most likely AI-Generated").sum()
    human_pred = (df["binoculars_prediction"] == "Most likely Human-Generated").sum()
    total = len(df)

    return {
        "label": label,
        "total_chunks": total,
        "scored_chunks": len(valid),
        "mean_score": valid.mean(),
        "median_score": valid.median(),
        "std_score": valid.std(),
        "ai_generated_count": ai_pred,
        "ai_generated_pct": ai_pred / total * 100,
        "human_generated_count": human_pred,
        "human_generated_pct": human_pred / total * 100,
        "unique_papers": df["paper_id"].nunique(),
    }

s21 = summarize_scores(df_2021_scored, "Dec 2021")
s25 = summarize_scores(df_2025_scored, "May 2024")

print(f"\n{'Metric':<30} {'Dec 2021':>15} {'Dec 2025':>15} {'Delta':>15}")
print("-" * 80)
print(f"{'Total chunks':<30} {s21['total_chunks']:>15} {s25['total_chunks']:>15}")
print(f"{'Unique papers':<30} {s21['unique_papers']:>15} {s25['unique_papers']:>15}")
print(f"{'Mean Binoculars score':<30} {s21['mean_score']:>15.4f} {s25['mean_score']:>15.4f} {s25['mean_score']-s21['mean_score']:>+15.4f}")
print(f"{'Median Binoculars score':<30} {s21['median_score']:>15.4f} {s25['median_score']:>15.4f} {s25['median_score']-s21['median_score']:>+15.4f}")
print(f"{'Std dev of score':<30} {s21['std_score']:>15.4f} {s25['std_score']:>15.4f}")
print(f"{'Classified AI-generated':<30} {s21['ai_generated_count']:>11} ({s21['ai_generated_pct']:.1f}%) {s25['ai_generated_count']:>11} ({s25['ai_generated_pct']:.1f}%) {s25['ai_generated_pct']-s21['ai_generated_pct']:>+12.1f}pp")
print(f"{'Classified Human-generated':<30} {s21['human_generated_count']:>11} ({s21['human_generated_pct']:.1f}%) {s25['human_generated_count']:>11} ({s25['human_generated_pct']:.1f}%)")

# --- Per-paper aggregation ---
print("\n\nPer-Paper Aggregated Scores:")
print("-" * 80)

for df_scored, label in [(df_2021_scored, "Dec 2021"), (df_2025_scored, "Dec 2025")]:
    paper_scores = df_scored.groupby("paper_id")["binoculars_score"].agg(["mean", "median", "std", "count"])
    paper_ai_pct = df_scored.groupby("paper_id").apply(
        lambda g: (g["binoculars_prediction"] == "Most likely AI-Generated").mean() * 100
    )

    # Papers where majority of chunks are classified as AI
    majority_ai = (paper_ai_pct > 50).sum()
    all_ai = (paper_ai_pct == 100).sum()

    print(f"\n  {label}:")
    print(f"    Papers: {len(paper_scores)}")
    print(f"    Mean paper score:      {paper_scores['mean'].mean():.4f}")
    print(f"    Papers majority AI:    {majority_ai} ({majority_ai/len(paper_scores)*100:.1f}%)")
    print(f"    Papers 100% AI chunks: {all_ai} ({all_ai/len(paper_scores)*100:.1f}%)")

# --- Save combined summary ---
summary = {
    "run_timestamp": datetime.now().isoformat(),
    "config": {
        "batch_size": BATCH_SIZE,
        "use_fp16": USE_FP16,
        "binoculars_models": "Falcon-7B + Falcon-7B-Instruct (default)",
    },
    "dec_2021": s21,
    "dec_2025": s25,
}

summary_path = os.path.join(OUTPUT_DIR, "scoring_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n\nSummary saved to: {summary_path}")
print(f"\nScored CSVs saved to:")
print(f"  {os.path.join(OUTPUT_DIR, 'scored_dec_2021.csv')}")
print(f"  {os.path.join(OUTPUT_DIR, 'scored_dec_2025.csv')}")


# =============================================================================
# CELL 7: Quick visualization (optional)
# =============================================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Binoculars Score Distributions", fontsize=16, fontweight="bold")

C_2021 = "#2E86AB"
C_2025 = "#E84855"

# Score distributions
ax = axes[0]
scores_21 = df_2021_scored["binoculars_score"].dropna()
scores_25 = df_2025_scored["binoculars_score"].dropna()
ax.hist(scores_21, bins=50, alpha=0.6, color=C_2021, label="Dec 2021", density=True, edgecolor="white", linewidth=0.3)
ax.hist(scores_25, bins=50, alpha=0.6, color=C_2025, label="Dec 2025", density=True, edgecolor="white", linewidth=0.3)
ax.axvline(x=0.9015, color="black", linestyle="--", linewidth=1, label="Threshold (0.9015)")
ax.set_xlabel("Binoculars Score")
ax.set_ylabel("Density")
ax.set_title("Score Distribution (Chunk-Level)")
ax.legend(frameon=False)

# AI classification rates
ax = axes[1]
labels = ["Dec 2021\n(Pre-LLM)", "Dec 2025\n(Post-LLM)"]
ai_rates = [s21["ai_generated_pct"], s25["ai_generated_pct"]]
human_rates = [s21["human_generated_pct"], s25["human_generated_pct"]]

x = range(len(labels))
ax.bar(x, ai_rates, color=[C_2021, C_2025], edgecolor="white", width=0.5)
for i, (rate, label) in enumerate(zip(ai_rates, labels)):
    ax.text(i, rate + 1, f"{rate:.1f}%", ha="center", fontweight="bold", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("% Classified as AI-Generated")
ax.set_title("AI Detection Rate by Period")
ax.set_ylim(0, max(ai_rates) * 1.3)

plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, "binoculars_results_overview.png")
fig.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figure saved to: {fig_path}")
plt.show()

In [ ]:
!pip install transformers==4.38.0 --quiet

In [ ]:
print(df_2021_scored["binoculars_score"].describe())
print(df_2025_scored["binoculars_score"].describe())

In [ ]:
# Most AI-like chunks from 2021 (these are likely false positives)
print("=== Most AI-like from 2021 (false positives?) ===")
for _, row in df_2021_scored.nsmallest(5, "binoculars_score").iterrows():
    print(f"\nScore: {row['binoculars_score']:.4f} | Paper: {row['paper_id']}")
    print(row["text"][:300])

# Most human-like chunks from 2025
print("\n=== Most human-like from 2025 ===")
for _, row in df_2025_scored.nlargest(5, "binoculars_score").iterrows():
    print(f"\nScore: {row['binoculars_score']:.4f} | Paper: {row['paper_id']}")
    print(row["text"][:300])

# Most AI-like chunks from 2025
print("\n=== Most AI-like from 2025 ===")
for _, row in df_2025_scored.nsmallest(5, "binoculars_score").iterrows():
    print(f"\nScore: {row['binoculars_score']:.4f} | Paper: {row['paper_id']}")
    print(row["text"][:300])

In [ ]:
thresholds = [0.75, 0.80, 0.85, 0.90, 0.9015, 0.95, 1.00, 1.05, 1.10]
print(f"{'Threshold':<12} {'2021 AI %':>10} {'2025 AI %':>10} {'Difference':>12}")
print("-" * 48)
for t in thresholds:
    rate_21 = (df_2021_scored["binoculars_score"] < t).mean() * 100
    rate_25 = (df_2025_scored["binoculars_score"] < t).mean() * 100
    print(f"{t:<12.4f} {rate_21:>9.1f}% {rate_25:>9.1f}% {rate_25-rate_21:>+10.1f}pp")

## Positive Control

In [ ]:
!pip install openai --quiet

In [ ]:
import os
from google.colab import userdata

# REDACTED — original Anthropic + OpenAI keys were rotated and removed.
# Set your own via Colab Secrets, then uncomment whichever provider you want to use.
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
# os.environ['OPENAI_API_KEY']    = userdata.get('OPENAI_API_KEY')


#Hide

In [ ]:
"""
Positive Control Experiment: LLM Rewriting + Binoculars Re-scoring
===================================================================
Takes a sample of 2021 (pre-LLM) chunks, rewrites them through an LLM,
then scores both originals and rewrites with Binoculars to verify the
detector can actually detect LLM-generated academic text.

This uses the Anthropic API (Claude) for rewriting, which is available
in Colab. You can also swap in OpenAI or any other LLM API.

Setup:
    !pip install anthropic
    # Set your API key:
    # import os; os.environ["ANTHROPIC_API_KEY"] = "your-key-here"

    # OR use OpenAI:
    # !pip install openai
    # import os; os.environ["OPENAI_API_KEY"] = "your-key-here"
"""

# =============================================================================
# CELL 1: Configuration
# =============================================================================

import os
import json
import time
import random
import pandas as pd
import numpy as np
from datetime import datetime

# --- CONFIGURATION ---
# Input: scored 2021 CSV (from the Binoculars pipeline)
SCORED_2021_CSV = "/content/drive/MyDrive/arxiv/binoculars_results/scored_dec_2021.csv"  # <-- CHANGE

# Output directory
OUTPUT_DIR = "/content/drive/MyDrive/arxiv/positive_control_gpt4o"

# Sample size — how many chunks to rewrite
# Keep this manageable since each chunk = 1 API call
SAMPLE_SIZE = 40

# Which LLM API to use: "anthropic" or "openai"
LLM_PROVIDER = "openai"  # <-- CHANGE if using OpenAI

# Random seed for reproducibility
RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Load the scored 2021 data
df_2021 = pd.read_csv(SCORED_2021_CSV)
print(f"Loaded {len(df_2021)} chunks from 2021 dataset")

# Sample chunks — stratify by score to get a representative sample
# (don't just pick the lowest/highest scoring ones)
df_sample = df_2021.dropna(subset=["binoculars_score"]).sample(
    n=min(SAMPLE_SIZE, len(df_2021)),
    random_state=RANDOM_SEED
).reset_index(drop=True)

print(f"Sampled {len(df_sample)} chunks for rewriting")
print(f"Original score stats: mean={df_sample['binoculars_score'].mean():.4f}, "
      f"median={df_sample['binoculars_score'].median():.4f}")


# =============================================================================
# CELL 2: Define rewriting functions
# =============================================================================

# --- Rewriting prompts ---
# We test multiple rewriting scenarios to see which ones Binoculars catches

REWRITE_PROMPTS = {
    "full_rewrite": {
    "system": "You are an academic researcher writing a paper.",
    "user": (
        "Based on the following passage from a research paper, write a new "
        "introduction paragraph for this paper. The introduction should "
        "motivate the research problem, briefly describe the approach, and "
        "hint at the findings. Write approximately 256 words. Write naturally "
        "as an academic researcher would. Output ONLY the introduction "
        "with no preamble or explanation.\n\n"
        "Reference passage:\n{text}"
    ),
},
    "polish": {
    "system": "You are an academic researcher writing a paper.",
    "user": (
        "Completely rewrite the following academic passage. Change the "
        "sentence structure, reorganize the arguments, and use different "
        "vocabulary while preserving the technical meaning. The result "
        "should read as if a different author wrote about the same topic. "
        "Output ONLY the rewritten text with no preamble or explanation.\n\n"
        "Original text:\n{text}"
    ),
},
    "expand": {
        "system": "You are a helpful academic writing assistant.",
        "user": (
            "Rewrite and slightly expand the following academic text passage. "
            "Add connecting phrases, improve transitions, and make the writing "
            "more fluent. Keep the same technical content and meaning. "
            "Output ONLY the rewritten text with no preamble or explanation.\n\n"
            "Original text:\n{text}"
        ),
    },
}

# Select which rewrite modes to run
REWRITE_MODES = ["full_rewrite", "polish", "expand"]


def rewrite_with_anthropic(text: str, mode: str) -> str:
    """Rewrite text using the Anthropic API."""
    import anthropic
    client = anthropic.Anthropic()  # Uses ANTHROPIC_API_KEY env var

    prompt_config = REWRITE_PROMPTS[mode]

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        system=prompt_config["system"],
        messages=[
            {"role": "user", "content": prompt_config["user"].format(text=text)}
        ],
    )

    return response.content[0].text.strip()


def rewrite_with_openai(text: str, mode: str) -> str:
    """Rewrite text using the OpenAI API."""
    from openai import OpenAI
    client = OpenAI()  # Uses OPENAI_API_KEY env var

    prompt_config = REWRITE_PROMPTS[mode]

    response = client.chat.completions.create(
        model="gpt-4o",
        max_tokens=1024,
        messages=[
            {"role": "system", "content": prompt_config["system"]},
            {"role": "user", "content": prompt_config["user"].format(text=text)},
        ],
    )

    return response.choices[0].message.content.strip()


# Select rewrite function based on provider
if LLM_PROVIDER == "anthropic":
    rewrite_fn = rewrite_with_anthropic
    print("Using Anthropic API (Claude) for rewriting")
elif LLM_PROVIDER == "openai":
    rewrite_fn = rewrite_with_openai
    print("Using OpenAI API (GPT-4o) for rewriting")
else:
    raise ValueError(f"Unknown LLM provider: {LLM_PROVIDER}")


# =============================================================================
# CELL 3: Run the rewrites
# =============================================================================

# Checkpoint support
checkpoint_path = os.path.join(OUTPUT_DIR, "rewrite_checkpoint.json")
rewrite_results = []
start_idx = 0

if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r") as f:
        checkpoint = json.load(f)
    rewrite_results = checkpoint["results"]
    start_idx = checkpoint["next_idx"]
    print(f"Resuming from checkpoint: {start_idx}/{len(df_sample)} chunks done")

total_to_process = len(df_sample) * len(REWRITE_MODES)
processed = len(rewrite_results)

print(f"\nRewriting {len(df_sample)} chunks × {len(REWRITE_MODES)} modes = {total_to_process} total rewrites")
print(f"Already done: {processed}")
print("This will take a while — each rewrite is an API call...\n")

rewrite_start = time.time()

for i in range(start_idx, len(df_sample)):
    row = df_sample.iloc[i]
    original_text = str(row["text"])

    for mode in REWRITE_MODES:
        try:
            rewritten = rewrite_fn(original_text, mode)

            rewrite_results.append({
                "paper_id": str(row["paper_id"]),
                "chunk_index": int(row.get("chunk_index", i)),
                "original_text": original_text,
                "rewritten_text": rewritten,
                "rewrite_mode": mode,
                "original_binoculars_score": float(row["binoculars_score"]),
                "original_word_count": int(len(original_text.split())),
                "rewritten_word_count": int(len(rewritten.split())),
            })

            processed += 1

        except Exception as e:
            print(f"  Error on chunk {i}, mode {mode}: {e}")
            rewrite_results.append({
                "paper_id": str(row["paper_id"]),
                "chunk_index": int(row.get("chunk_index", i)),
                "original_text": original_text,
                "rewritten_text": None,
                "rewrite_mode": mode,
                "original_binoculars_score": float(row["binoculars_score"]),
                "error": str(e),
            })
            processed += 1

        # Rate limiting — small delay between API calls
        time.sleep(0.5)

    # Save checkpoint after each chunk (all modes)
    with open(checkpoint_path, "w") as f:
        json.dump({"results": rewrite_results, "next_idx": int(i + 1)}, f)

    # Progress
    if (i - start_idx) % 10 == 0:
        elapsed = time.time() - rewrite_start
        rate = (i - start_idx + 1) / elapsed * 60 if elapsed > 0 else 0
        remaining = (len(df_sample) - i - 1) / rate if rate > 0 else 0
        print(f"  [{i+1}/{len(df_sample)}] {rate:.1f} chunks/min | ETA: {remaining:.1f} min")

total_time = time.time() - rewrite_start
print(f"\nRewriting complete: {processed} rewrites in {total_time/60:.1f} minutes")

# Save intermediate results
rewrite_df = pd.DataFrame(rewrite_results)
rewrite_csv = os.path.join(OUTPUT_DIR, "rewritten_chunks.csv")
rewrite_df.to_csv(rewrite_csv, index=False)
print(f"Saved rewrites to: {rewrite_csv}")

# Clean up checkpoint
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)


# =============================================================================
# CELL 4: Score the rewritten chunks with Binoculars
# =============================================================================

import torch

# Load Binoculars (if not already loaded from the inference pipeline)
# If already loaded in another cell, skip this
try:
    bino
    print("Binoculars already loaded")
except NameError:
    import sys
    sys.path.insert(0, "/content/Binoculars")
    torch.set_default_dtype(torch.float16)
    from binoculars import Binoculars
    bino = Binoculars()
    torch.set_default_dtype(torch.float32)
    print("Binoculars loaded")

# Load the rewrite results
rewrite_df = pd.read_csv(os.path.join(OUTPUT_DIR, "rewritten_chunks.csv"))

# Filter out failed rewrites
valid_rewrites = rewrite_df.dropna(subset=["rewritten_text"]).copy()
print(f"\nScoring {len(valid_rewrites)} rewritten chunks with Binoculars...")

# Batch score the rewritten texts
BATCH_SIZE = 8
rewritten_scores = []

for i in range(0, len(valid_rewrites), BATCH_SIZE):
    batch = valid_rewrites.iloc[i:i+BATCH_SIZE]
    texts = [str(t) for t in batch["rewritten_text"].tolist()]

    try:
        scores = bino.compute_score(texts)
        preds = bino.predict(texts)

        if isinstance(scores, (int, float)):
            scores = [scores]
            preds = [preds]

        for s, p in zip(scores, preds):
            rewritten_scores.append({"score": float(s), "prediction": str(p)})
    except Exception as e:
        # Fallback to individual scoring
        for t in texts:
            try:
                s = bino.compute_score(t)
                p = bino.predict(t)
                rewritten_scores.append({"score": float(s), "prediction": str(p)})
            except:
                rewritten_scores.append({"score": None, "prediction": "error"})

    if i % (BATCH_SIZE * 10) == 0:
        print(f"  [{i+len(batch)}/{len(valid_rewrites)}]")

# Add scores to dataframe
scores_df = pd.DataFrame(rewritten_scores)
valid_rewrites = valid_rewrites.reset_index(drop=True)
valid_rewrites["rewritten_binoculars_score"] = scores_df["score"]
valid_rewrites["rewritten_binoculars_prediction"] = scores_df["prediction"]

# Save final results
final_csv = os.path.join(OUTPUT_DIR, "positive_control_results.csv")
valid_rewrites.to_csv(final_csv, index=False)
print(f"\nResults saved to: {final_csv}")


# =============================================================================
# CELL 5: Analyze results
# =============================================================================

import matplotlib.pyplot as plt

results = pd.read_csv(os.path.join(OUTPUT_DIR, "positive_control_results.csv"))
results = results.dropna(subset=["rewritten_binoculars_score", "original_binoculars_score"])

print("=" * 70)
print("POSITIVE CONTROL RESULTS")
print("=" * 70)

# --- Per-mode analysis ---
print(f"\n{'Mode':<15} {'Orig Score':>12} {'Rewrite Score':>14} {'Delta':>10} {'Orig AI%':>10} {'Rewrite AI%':>12}")
print("-" * 80)

for mode in REWRITE_MODES:
    subset = results[results["rewrite_mode"] == mode]
    orig_mean = subset["original_binoculars_score"].mean()
    rewrite_mean = subset["rewritten_binoculars_score"].mean()
    delta = rewrite_mean - orig_mean

    orig_ai = (subset["original_binoculars_score"] < 0.9015).mean() * 100
    rewrite_ai = (subset["rewritten_binoculars_score"] < 0.9015).mean() * 100

    print(f"{mode:<15} {orig_mean:>12.4f} {rewrite_mean:>14.4f} {delta:>+10.4f} {orig_ai:>9.1f}% {rewrite_ai:>11.1f}%")

# --- Overall summary ---
orig_mean = results["original_binoculars_score"].mean()
rewrite_mean = results["rewritten_binoculars_score"].mean()
print(f"\n{'OVERALL':<15} {orig_mean:>12.4f} {rewrite_mean:>14.4f} {rewrite_mean-orig_mean:>+10.4f}")

# --- Multi-threshold analysis ---
print(f"\n\nMulti-Threshold Detection Rates:")
print(f"{'Threshold':<12} {'Original AI%':>14} {'Rewrite AI%':>14} {'Difference':>12}")
print("-" * 55)

thresholds = [0.80, 0.85, 0.90, 0.9015, 0.95, 1.00, 1.05, 1.10]
for t in thresholds:
    orig_rate = (results["original_binoculars_score"] < t).mean() * 100
    rewrite_rate = (results["rewritten_binoculars_score"] < t).mean() * 100
    print(f"{t:<12.4f} {orig_rate:>13.1f}% {rewrite_rate:>13.1f}% {rewrite_rate-orig_rate:>+10.1f}pp")

# --- Statistical test ---
from scipy import stats

for mode in REWRITE_MODES:
    subset = results[results["rewrite_mode"] == mode]
    orig = subset["original_binoculars_score"]
    rewrite = subset["rewritten_binoculars_score"]

    # Paired test since each rewrite corresponds to a specific original
    t_stat, p_value = stats.ttest_rel(orig, rewrite)
    cohens_d = (rewrite.mean() - orig.mean()) / np.sqrt((orig.std()**2 + rewrite.std()**2) / 2)

    print(f"\n  {mode}: paired t-test t={t_stat:.3f}, p={p_value:.2e}, Cohen's d={cohens_d:.3f}")


# --- Visualization ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Positive Control: Original vs LLM-Rewritten Academic Text",
             fontsize=16, fontweight="bold", y=1.02)

C_ORIG = "#2E86AB"
C_REWRITE = "#E84855"

# 1: Score distributions by mode
ax = axes[0][0]
for mode in REWRITE_MODES:
    subset = results[results["rewrite_mode"] == mode]
    ax.hist(subset["rewritten_binoculars_score"].dropna(), bins=30, alpha=0.4,
            label=f"Rewrite ({mode})", density=True, edgecolor="white", linewidth=0.3)
ax.hist(results["original_binoculars_score"].dropna(), bins=30, alpha=0.6,
        color=C_ORIG, label="Original (human)", density=True, edgecolor="white", linewidth=0.3)
ax.axvline(x=0.9015, color="black", linestyle="--", linewidth=1, label="Threshold")
ax.set_xlabel("Binoculars Score")
ax.set_ylabel("Density")
ax.set_title("Score Distributions: Originals vs Rewrites")
ax.legend(frameon=False, fontsize=8)

# 2: Paired scatter — original vs rewritten scores
ax = axes[0][1]
for mode, marker, color in zip(REWRITE_MODES, ["o", "s", "^"], ["#E84855", "#FF9F1C", "#7B2D8E"]):
    subset = results[results["rewrite_mode"] == mode]
    ax.scatter(subset["original_binoculars_score"], subset["rewritten_binoculars_score"],
               alpha=0.3, s=15, marker=marker, color=color, label=mode)
lims = [
    min(results["original_binoculars_score"].min(), results["rewritten_binoculars_score"].min()) - 0.05,
    max(results["original_binoculars_score"].max(), results["rewritten_binoculars_score"].max()) + 0.05,
]
ax.plot(lims, lims, "k--", alpha=0.3, linewidth=1, label="y=x (no change)")
ax.axhline(y=0.9015, color="gray", linestyle=":", alpha=0.5)
ax.axvline(x=0.9015, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Original Score")
ax.set_ylabel("Rewritten Score")
ax.set_title("Paired Comparison: Original vs Rewritten")
ax.legend(frameon=False, fontsize=8)

# 3: Score shift by mode (box plot)
ax = axes[1][0]
shift_data = []
shift_labels = []
for mode in REWRITE_MODES:
    subset = results[results["rewrite_mode"] == mode]
    shift = subset["rewritten_binoculars_score"].values - subset["original_binoculars_score"].values
    shift_data.append(shift)
    shift_labels.append(mode)

bp = ax.boxplot(shift_data, labels=shift_labels, patch_artist=True, widths=0.6)
colors = ["#E84855", "#FF9F1C", "#7B2D8E"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.axhline(y=0, color="black", linestyle="--", linewidth=1)
ax.set_ylabel("Score Shift (rewrite - original)")
ax.set_title("Score Change by Rewrite Mode")

# 4: Detection rate comparison
ax = axes[1][1]
mode_labels = []
orig_rates = []
rewrite_rates = []
for mode in REWRITE_MODES:
    subset = results[results["rewrite_mode"] == mode]
    mode_labels.append(mode.replace("_", "\n"))
    orig_rates.append((subset["original_binoculars_score"] < 0.9015).mean() * 100)
    rewrite_rates.append((subset["rewritten_binoculars_score"] < 0.9015).mean() * 100)

x = np.arange(len(mode_labels))
width = 0.35
ax.bar(x - width/2, orig_rates, width, color=C_ORIG, label="Original", edgecolor="white")
ax.bar(x + width/2, rewrite_rates, width, color=C_REWRITE, label="Rewritten", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(mode_labels)
ax.set_ylabel("% Classified as AI")
ax.set_title("AI Detection Rate (threshold=0.9015)")
ax.legend(frameon=False)

# Add percentage labels
for i, (o, r) in enumerate(zip(orig_rates, rewrite_rates)):
    ax.text(i - width/2, o + 0.5, f"{o:.1f}%", ha="center", fontsize=9)
    ax.text(i + width/2, r + 0.5, f"{r:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "positive_control_results.png")
fig.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"\nFigure saved to: {fig_path}")
plt.show()

# --- Key takeaway ---
print("\n" + "=" * 70)
print("KEY FINDINGS")
print("=" * 70)

overall_orig_ai = (results["original_binoculars_score"] < 0.9015).mean() * 100
overall_rewrite_ai = (results["rewritten_binoculars_score"] < 0.9015).mean() * 100

print(f"\nOriginal human text flagged as AI:    {overall_orig_ai:.1f}%")
print(f"LLM-rewritten text flagged as AI:     {overall_rewrite_ai:.1f}%")
print(f"Difference:                           {overall_rewrite_ai - overall_orig_ai:+.1f}pp")

if overall_rewrite_ai > overall_orig_ai + 10:
    print("\n→ Binoculars CAN detect LLM-rewritten academic text.")
    print("  This means the low detection rate in 2025 data is meaningful —")
    print("  it suggests LLM usage in actual papers is either low or involves")
    print("  lighter editing that doesn't leave a strong signal.")
elif overall_rewrite_ai > overall_orig_ai + 2:
    print("\n→ Binoculars shows MODERATE sensitivity to LLM-rewritten academic text.")
    print("  The detector picks up some signal but misses a lot.")
    print("  This suggests detection in academic domains is harder than in")
    print("  news/essays, and your main results should be interpreted with caution.")
else:
    print("\n→ Binoculars CANNOT reliably detect LLM-rewritten academic text.")
    print("  The detector performs similarly on original and rewritten text.")
    print("  This means the main experiment's null result may reflect detector")
    print("  limitations rather than absence of LLM usage.")

#Unhide

In [ ]:
%cd /content/Binoculars
!pip install -e . --quiet

In [ ]:
%cd /content/Binoculars
!pip install --no-deps -e . --quiet
!pip install transformers==4.38.0 --quiet
!pip install datasets numpy scikit-learn --quiet

In [ ]:
# Check what's actually in the directory
!ls /content/Binoculars/

# Check if the binoculars package folder exists
!ls /content/Binoculars/binoculars/

# Check if pip knows about it
!pip show Binoculars

# Check where Python is looking for modules
import sys
print([p for p in sys.path if 'Binoculars' in p or 'binoculars' in p])

In [ ]:
import importlib
import sys

# Clear any cached failed imports
if 'binoculars' in sys.modules:
    del sys.modules['binoculars']

# Make sure the path is there
if '/content/Binoculars' not in sys.path:
    sys.path.insert(0, '/content/Binoculars')

from binoculars import Binoculars

In [ ]:
"""
Re-score rewritten chunks with Binoculars
==========================================
Run this after the rewrites are already saved to CSV.
"""

# =============================================================================
# CELL 1: Setup and load Binoculars
# =============================================================================

import os
import sys
import torch
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
REWRITE_CSV = "/content/drive/MyDrive/arxiv/positive_control_gpt4o/rewritten_chunks.csv"  # <-- CHANGE
OUTPUT_DIR = "/content/drive/MyDrive/arxiv/positive_control_gpt4o"  # <-- CHANGE
BATCH_SIZE = 8

os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.set_default_dtype(torch.float16)
from binoculars import Binoculars
print("Loading Binoculars models...")
bino = Binoculars()
torch.set_default_dtype(torch.float32)
print("Binoculars loaded!")

# Sanity check
test = bino.compute_score("This is a test sentence.")
print(f"Sanity check score: {test:.4f}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


# =============================================================================
# CELL 2: Score the rewritten chunks
# =============================================================================

import time

# Load rewrite results
rewrite_df = pd.read_csv(REWRITE_CSV)
print(f"Loaded {len(rewrite_df)} rows from {REWRITE_CSV}")

# Filter out failed rewrites
valid = rewrite_df.dropna(subset=["rewritten_text"]).copy()
failed = len(rewrite_df) - len(valid)
if failed > 0:
    print(f"Skipping {failed} failed rewrites")
print(f"Scoring {len(valid)} rewritten chunks...\n")

# Score in batches
rewritten_scores = []
rewritten_preds = []
score_start = time.time()

for i in range(0, len(valid), BATCH_SIZE):
    batch_end = min(i + BATCH_SIZE, len(valid))
    texts = [str(t) for t in valid["rewritten_text"].iloc[i:batch_end].tolist()]

    try:
        scores = bino.compute_score(texts)
        preds = bino.predict(texts)

        if isinstance(scores, (int, float)):
            scores = [scores]
            preds = [preds]

        rewritten_scores.extend([float(s) for s in scores])
        rewritten_preds.extend([str(p) for p in preds])

    except Exception as e:
        print(f"  Batch error at {i}, falling back to individual scoring: {e}")
        for t in texts:
            try:
                s = bino.compute_score(t)
                p = bino.predict(t)
                rewritten_scores.append(float(s))
                rewritten_preds.append(str(p))
            except Exception as inner_e:
                rewritten_scores.append(None)
                rewritten_preds.append(f"error: {str(inner_e)[:50]}")

    if i % (BATCH_SIZE * 10) == 0:
        elapsed = time.time() - score_start
        rate = (i + len(texts)) / elapsed if elapsed > 0 else 0
        eta = (len(valid) - i - len(texts)) / rate / 60 if rate > 0 else 0
        print(f"  [{i + len(texts):>5}/{len(valid)}] {rate:.1f} chunks/s | ETA: {eta:.1f} min")

# Add scores to dataframe
valid = valid.reset_index(drop=True)
valid["rewritten_binoculars_score"] = rewritten_scores
valid["rewritten_binoculars_prediction"] = rewritten_preds

# Save
output_csv = os.path.join(OUTPUT_DIR, "positive_control_results.csv")
valid.to_csv(output_csv, index=False)

total_time = time.time() - score_start
print(f"\nDone! Scored {len(valid)} chunks in {total_time / 60:.1f} minutes")
print(f"Saved to: {output_csv}")


# =============================================================================
# CELL 3: Analyze results
# =============================================================================

import matplotlib.pyplot as plt
from scipy import stats

results = pd.read_csv(output_csv)
results = results.dropna(subset=["rewritten_binoculars_score", "original_binoculars_score"])

REWRITE_MODES = results["rewrite_mode"].unique().tolist()

print("=" * 70)
print("POSITIVE CONTROL RESULTS")
print("=" * 70)

# --- Per-mode analysis ---
print(f"\n{'Mode':<15} {'Orig Score':>12} {'Rewrite Score':>14} {'Delta':>10} {'Orig AI%':>10} {'Rewrite AI%':>12}")
print("-" * 80)

for mode in REWRITE_MODES:
    subset = results[results["rewrite_mode"] == mode]
    orig_mean = subset["original_binoculars_score"].mean()
    rewrite_mean = subset["rewritten_binoculars_score"].mean()
    delta = rewrite_mean - orig_mean

    orig_ai = (subset["original_binoculars_score"] < 0.9015).mean() * 100
    rewrite_ai = (subset["rewritten_binoculars_score"] < 0.9015).mean() * 100

    print(f"{mode:<15} {orig_mean:>12.4f} {rewrite_mean:>14.4f} {delta:>+10.4f} {orig_ai:>9.1f}% {rewrite_ai:>11.1f}%")

# --- Overall ---
orig_mean = results["original_binoculars_score"].mean()
rewrite_mean = results["rewritten_binoculars_score"].mean()
print(f"\n{'OVERALL':<15} {orig_mean:>12.4f} {rewrite_mean:>14.4f} {rewrite_mean - orig_mean:>+10.4f}")

# --- Multi-threshold analysis ---
print(f"\n\nMulti-Threshold Detection Rates:")
print(f"{'Threshold':<12} {'Original AI%':>14} {'Rewrite AI%':>14} {'Difference':>12}")
print("-" * 55)

thresholds = [0.80, 0.85, 0.90, 0.9015, 0.95, 1.00, 1.05, 1.10]
for t in thresholds:
    orig_rate = (results["original_binoculars_score"] < t).mean() * 100
    rewrite_rate = (results["rewritten_binoculars_score"] < t).mean() * 100
    print(f"{t:<12.4f} {orig_rate:>13.1f}% {rewrite_rate:>13.1f}% {rewrite_rate - orig_rate:>+10.1f}pp")

# --- Statistical tests ---
print("\n\nStatistical Tests (paired):")
print("-" * 70)
for mode in REWRITE_MODES:
    subset = results[results["rewrite_mode"] == mode]
    orig = subset["original_binoculars_score"]
    rewrite = subset["rewritten_binoculars_score"]

    t_stat, p_value = stats.ttest_rel(orig, rewrite)
    cohens_d = (rewrite.mean() - orig.mean()) / np.sqrt((orig.std() ** 2 + rewrite.std() ** 2) / 2)

    print(f"  {mode}: t={t_stat:.3f}, p={p_value:.2e}, Cohen's d={cohens_d:.3f}")

# --- Visualization ---
C_ORIG = "#2E86AB"
C_REWRITE = "#E84855"

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Positive Control: Original vs LLM-Rewritten Academic Text",
             fontsize=16, fontweight="bold", y=1.02)

# 1: Score distributions
ax = axes[0][0]
for mode in REWRITE_MODES:
    subset = results[results["rewrite_mode"] == mode]
    ax.hist(subset["rewritten_binoculars_score"].dropna(), bins=30, alpha=0.4,
            label=f"Rewrite ({mode})", density=True, edgecolor="white", linewidth=0.3)
ax.hist(results["original_binoculars_score"].dropna(), bins=30, alpha=0.6,
        color=C_ORIG, label="Original (human)", density=True, edgecolor="white", linewidth=0.3)
ax.axvline(x=0.9015, color="black", linestyle="--", linewidth=1, label="Threshold")
ax.set_xlabel("Binoculars Score")
ax.set_ylabel("Density")
ax.set_title("Score Distributions: Originals vs Rewrites")
ax.legend(frameon=False, fontsize=8)

# 2: Paired scatter
ax = axes[0][1]
for mode, marker, color in zip(REWRITE_MODES, ["o", "s", "^"], ["#E84855", "#FF9F1C", "#7B2D8E"]):
    subset = results[results["rewrite_mode"] == mode]
    ax.scatter(subset["original_binoculars_score"], subset["rewritten_binoculars_score"],
               alpha=0.3, s=15, marker=marker, color=color, label=mode)
lims = [
    min(results["original_binoculars_score"].min(), results["rewritten_binoculars_score"].min()) - 0.05,
    max(results["original_binoculars_score"].max(), results["rewritten_binoculars_score"].max()) + 0.05,
]
ax.plot(lims, lims, "k--", alpha=0.3, linewidth=1, label="y=x (no change)")
ax.axhline(y=0.9015, color="gray", linestyle=":", alpha=0.5)
ax.axvline(x=0.9015, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Original Score")
ax.set_ylabel("Rewritten Score")
ax.set_title("Paired Comparison: Original vs Rewritten")
ax.legend(frameon=False, fontsize=8)

# 3: Score shift box plot
ax = axes[1][0]
shift_data = []
for mode in REWRITE_MODES:
    subset = results[results["rewrite_mode"] == mode]
    shift = subset["rewritten_binoculars_score"].values - subset["original_binoculars_score"].values
    shift_data.append(shift)

bp = ax.boxplot(shift_data, labels=[m.replace("_", "\n") for m in REWRITE_MODES],
                patch_artist=True, widths=0.6)
colors = ["#E84855", "#FF9F1C", "#7B2D8E"]
for patch, color in zip(bp["boxes"], colors[:len(REWRITE_MODES)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.axhline(y=0, color="black", linestyle="--", linewidth=1)
ax.set_ylabel("Score Shift (rewrite − original)")
ax.set_title("Score Change by Rewrite Mode")

# 4: Detection rate comparison
ax = axes[1][1]
mode_labels = []
orig_rates = []
rewrite_rates = []
for mode in REWRITE_MODES:
    subset = results[results["rewrite_mode"] == mode]
    mode_labels.append(mode.replace("_", "\n"))
    orig_rates.append((subset["original_binoculars_score"] < 0.9015).mean() * 100)
    rewrite_rates.append((subset["rewritten_binoculars_score"] < 0.9015).mean() * 100)

x = np.arange(len(mode_labels))
width = 0.35
ax.bar(x - width / 2, orig_rates, width, color=C_ORIG, label="Original", edgecolor="white")
ax.bar(x + width / 2, rewrite_rates, width, color=C_REWRITE, label="Rewritten", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(mode_labels)
ax.set_ylabel("% Classified as AI")
ax.set_title("AI Detection Rate (threshold=0.9015)")
ax.legend(frameon=False)
for i, (o, r) in enumerate(zip(orig_rates, rewrite_rates)):
    ax.text(i - width / 2, o + 0.5, f"{o:.1f}%", ha="center", fontsize=9)
    ax.text(i + width / 2, r + 0.5, f"{r:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "positive_control_results.png")
fig.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"\nFigure saved to: {fig_path}")
plt.show()

# --- Key takeaway ---
overall_orig_ai = (results["original_binoculars_score"] < 0.9015).mean() * 100
overall_rewrite_ai = (results["rewritten_binoculars_score"] < 0.9015).mean() * 100

print("\n" + "=" * 70)
print("KEY FINDINGS")
print("=" * 70)
print(f"\nOriginal human text flagged as AI:    {overall_orig_ai:.1f}%")
print(f"LLM-rewritten text flagged as AI:     {overall_rewrite_ai:.1f}%")
print(f"Difference:                           {overall_rewrite_ai - overall_orig_ai:+.1f}pp")

if overall_rewrite_ai > overall_orig_ai + 10:
    print("\n→ Binoculars CAN detect LLM-rewritten academic text.")
    print("  The low detection rate in real 2025 data is meaningful.")
elif overall_rewrite_ai > overall_orig_ai + 2:
    print("\n→ Binoculars shows MODERATE sensitivity to LLM rewrites.")
    print("  The detector picks up some signal but misses a lot.")
else:
    print("\n→ Binoculars CANNOT reliably detect LLM-rewritten academic text.")
    print("  The null result may reflect detector limitations, not absence of LLM usage.")

In [ ]:
import os
checkpoint_path = "/content/drive/MyDrive/arxiv/positive_control/rewrite_checkpoint.json"
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print("Deleted corrupted checkpoint")

In [ ]:
!pip install anthropic --quiet